# DNS / DNSS Bayesiano — Gibbs Sampler con FFBS  *(v2)*
## Carter & Kohn (1994) · Posteriors Conjugadas · Tres modos de τ · Diagnósticos MCMC

### Modos de modelado de τ (`tau_mode`)

| `tau_mode` | Descripción | τ varía en t | Costo extra |
|---|---|---|---|
| `'estatico'` | Un τ global, posterior via MH (comportamiento original) | No | Ninguno |
| `'ventana'` | Gibbs en ventanas deslizantes — τ cambia entre ventanas | Por ventana | Moderado |
| `'ar1'` | τ_t sigue un proceso AR(1) latente, muestreado por FFBS auxiliar | Sí (continuo) | Alto |

### Estructura del Gibbs Sampler (por iteración s)

```
Modo 'estatico'  →  igual que v1: 5 bloques, tau MH al final
Modo 'ventana'   →  igual que v1 por ventana; tau se re-inicializa entre ventanas
Modo 'ar1'       →  Bloque extra: tau_1:T | theta → FFBS auxiliar en log-escala
                     tau_t = exp(x_t),  x_t = phi_x*(x_{t-1}-mu_x) + sigma_x*eps_t
```

### Referencias
Caldeira, Laurini & Portugal (2010) · Çakmaklı (2013) · Carter & Kohn (1994)
Kim & Nelson (1999) · Nakajima (2011) · Roberts & Rosenthal (2001)


## CELDA 0: Imports y configuración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from scipy.stats import invgamma, truncnorm, norm as sp_norm
from scipy import stats
from joblib import Parallel, delayed
import warnings, time
warnings.filterwarnings('ignore')

try:
    from numba import njit as _njit
    _NUMBA = True
    print("Numba disponible — FFBS JIT activado")
except ImportError:
    _NUMBA = False
    def _njit(**kw): return lambda f: f
    print("Numba no disponible — modo Python puro (5-10x mas lento)")

try:
    from statsmodels.graphics.tsaplots import plot_acf as _sm_acf
    _STATSMODELS = True
except ImportError:
    _STATSMODELS = False
    print("statsmodels no disponible — ACF manual activada")

plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.grid'        : True,
    'grid.alpha'       : 0.22,
    'font.size'        : 10,
    'axes.titlesize'   : 12,
    'axes.titleweight' : 'bold',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'legend.framealpha': 0.85,
    'legend.fontsize'  : 9,
})

COLORES_FACTOR  = ['#1f77b4', '#d62728', '#2ca02c', '#ff7f0e']
COLORES_CADENA  = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd',
                   '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']
COLORES_TAU     = ['#9467bd', '#8c564b']

np.random.seed(42)
print("Configuracion lista")


## CELDA 1: Carga de datos

In [ ]:
import os

RUTA_DATOS = r"C:\Users\ADMON\Documents\Universidad\Proyecto KMJ\codigo\TES_colombia\curvas_panel_historico_limpio.csv"

if os.path.exists(RUTA_DATOS):
    raw = pd.read_csv(RUTA_DATOS)
    yield_cols = ["SVENY01","SVENY02","SVENY03","SVENY05",
                  "SVENY07","SVENY10","SVENY20","SVENY30"]
    maturities = np.array([1, 2, 3, 5, 7, 10, 20, 30], dtype=float)
    df = raw[["Date"] + yield_cols].copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.dropna(subset=yield_cols).sort_values("Date").reset_index(drop=True)
    df = df[::10]   # ~mensual (cada 10 dias habiles)
    yields_data = df[yield_cols].values
    dates_dt    = pd.DatetimeIndex(df["Date"].values)
    fuente = "SVENY (Federal Reserve)"
else:
    # Datos sinteticos para prototipado
    T_sim  = 300
    maturities = np.array([1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])
    N_m = len(maturities)
    beta_true  = np.zeros((T_sim, 3))
    beta_true[0] = [5.5, -2.5, 1.8]
    phi_t = np.array([0.97, 0.94, 0.90])
    mu_t  = np.array([5.0, -2.0, 1.5])
    for t in range(1, T_sim):
        beta_true[t] = mu_t + phi_t*(beta_true[t-1]-mu_t) + np.random.randn(3)*np.array([0.07,0.13,0.20])
    tau_true = 1.8
    def _ns(m, tau):
        mt = m/tau; e = np.exp(-mt); c = (1-e)/mt
        return np.column_stack([np.ones_like(m), c, c-e])
    yields_data = np.zeros((T_sim, N_m))
    for t in range(T_sim):
        yields_data[t] = _ns(maturities, tau_true) @ beta_true[t] + np.random.randn(N_m)*0.07
    dates_dt   = pd.date_range("2005-01-01", periods=T_sim, freq="ME")
    yield_cols = [f"Y{int(m)}Y" for m in maturities]
    fuente = "Sintetico DNS con tau=1.8 (demostracion)"

T, N = yields_data.shape
print(f"Fuente : {fuente}")
print(f"T={T}  N={N}  plazos={maturities}")
print(f"Rango  : {dates_dt[0].date()} → {dates_dt[-1].date()}")


## CELDA 2: Cargas Nelson-Siegel y Svensson

In [ ]:
def ns_loadings(maturities, tau1):
    '''Cargas Nelson-Siegel. Shape (N, 3).'''
    m  = np.asarray(maturities, dtype=float)
    mt = m / tau1; e = np.exp(-mt); c = (1.0 - e) / mt
    return np.column_stack([np.ones_like(m), c, c - e])

def nss_loadings(maturities, tau1, tau2):
    '''Cargas Nelson-Siegel-Svensson. Shape (N, 4).'''
    m   = np.asarray(maturities, dtype=float)
    mt1 = m/tau1; mt2 = m/tau2
    e1  = np.exp(-mt1); e2 = np.exp(-mt2)
    c1  = (1.0 - e1) / mt1
    return np.column_stack([np.ones_like(m), c1, c1-e1, (1.0-e2)/mt2-e2])

def get_H(maturities, tau1, tau2, model):
    '''Wrapper unificado.'''
    if model == 'DNS':
        return ns_loadings(maturities, tau1)
    return nss_loadings(maturities, tau1, tau2)

def ols_init_tau(yields, maturities, model, tau1_grid=None, tau2_grid=None):
    '''Busca tau inicial via Grid OLS para arrancar el sampler.'''
    if tau1_grid is None:
        tau1_grid = np.linspace(0.5, 4.0, 12)
    if tau2_grid is None:
        tau2_grid = np.linspace(0.15, 2.0, 8)

    best_sse = np.inf
    best_t1, best_t2 = tau1_grid[len(tau1_grid)//2], tau2_grid[len(tau2_grid)//4]

    for t1 in tau1_grid:
        candidates = [None] if model == 'DNS' else tau2_grid
        for t2 in candidates:
            if model == 'DNSS' and abs(t1 - t2) < 0.15: continue
            H = get_H(maturities, t1, t2, model)
            try:
                B   = np.linalg.lstsq(H, yields.T, rcond=None)[0]
                sse = float(np.sum((yields.T - H @ B)**2))
            except Exception:
                continue
            if sse < best_sse:
                best_sse = sse; best_t1 = t1
                if model == 'DNSS': best_t2 = t2

    return best_t1, (best_t2 if model == 'DNSS' else None)

tau1_init, tau2_init = ols_init_tau(yields_data, maturities, model='DNS')
print(f"tau inicial OLS (DNS) : tau1={tau1_init:.2f}  hump~{1.793*tau1_init:.1f}Y")
tau1_init_nss, tau2_init_nss = ols_init_tau(yields_data, maturities, model='DNSS')
print(f"tau inicial OLS (DNSS): tau1={tau1_init_nss:.2f}  tau2={tau2_init_nss:.2f}")


## CELDA 3: Forward Filter Backward Sampler (FFBS) — JIT

El FFBS es la operación central del Gibbs sampler: muestrea la **trayectoria completa**
$\boldsymbol{\beta}_{1:T}$ condicionada en los parámetros actuales θ y las observaciones Y.
Esto es fundamental porque muestrear cada $\beta_t$ individualmente produce cadenas con
altísima autocorrelación — el FFBS elimina este problema.

**Forward (Kalman Filter):** computa $(\hat{\beta}_{t|t}, P_{t|t})$ para todo $t$.

**Backward:** muestrea hacia atrás:
$$\beta_T \sim \mathcal{N}(\hat{\beta}_{T|T},\ P_{T|T})$$
$$\beta_t | \beta_{t+1}, Y \sim \mathcal{N}(\mu_{t|T},\ \Sigma_{t|T})$$
$$\mu_{t|T} = \hat{\beta}_{t|t} + G_t(\beta_{t+1} - \hat{\beta}_{t+1|t}), \quad
  G_t = P_{t|t}F^\top P_{t+1|t}^{-1}$$

La implementación soporta tanto $k=3$ (DNS) como $k=4$ (DNSS) y F diagonal o general.


In [ ]:
if _NUMBA:
    @_njit(cache=True)
    def _ffbs_diag(mu, f_vec, q_vec, r_vec, H, yields):
        '''
        FFBS con F diagonal. Compilado JIT.
        Valido para k=3 (DNS) y k=4 (DNSS) segun el shape de mu/f_vec/H.
        Retorna: betas (T, k) — una trayectoria muestreada de p(beta_1:T | Y, theta).
        '''
        T_obs, N_obs = yields.shape
        k = len(mu)
        beta_filt = np.zeros((T_obs, k))
        P_filt    = np.zeros((T_obs, k, k))

        # Inicializacion estacionaria: P_0 = Q / (1 - f^2) por factor
        beta_hat = mu.copy()
        P = np.zeros((k, k))
        for i in range(k):
            P[i, i] = q_vec[i] / max(1.0 - f_vec[i]**2, 1e-6)
        Ik = np.eye(k)

        # ── FORWARD: Kalman Filter ────────────────────────────────────────────
        for t in range(T_obs):
            # Prediccion: beta_{t|t-1} = mu + F(beta_{t-1} - mu)
            beta_p = np.zeros(k)
            for i in range(k):
                beta_p[i] = mu[i] + f_vec[i] * (beta_hat[i] - mu[i])

            # P_{t|t-1} = F P F' + Q  (F diagonal: P_pred[i,j] = f_i P[i,j] f_j)
            P_p = np.zeros((k, k))
            for i in range(k):
                for j in range(k):
                    P_p[i, j] = f_vec[i] * P[i, j] * f_vec[j]
            for i in range(k):
                P_p[i, i] += q_vec[i]
            P_p = 0.5 * (P_p + P_p.T)

            # Innovacion y varianza
            v  = yields[t] - H @ beta_p
            HP = H @ P_p
            S  = HP @ H.T
            for n in range(N_obs):
                S[n, n] += r_vec[n]
            S = 0.5 * (S + S.T)

            # Ganancia de Kalman (via solucion lineal, no inversion)
            K = np.linalg.solve(S, HP).T    # (k, N)

            beta_hat = beta_p + K @ v

            # Actualizacion: forma Joseph
            IKH  = Ik - K @ H
            Rmat = np.zeros((N_obs, N_obs))
            for n in range(N_obs):
                Rmat[n, n] = r_vec[n]
            P = IKH @ P_p @ IKH.T + K @ Rmat @ K.T
            P = 0.5 * (P + P.T)

            beta_filt[t] = beta_hat
            P_filt[t]    = P.copy()

        # ── BACKWARD: Muestreo hacia atras ───────────────────────────────────
        betas = np.zeros((T_obs, k))

        # Muestra beta_T ~ N(beta_T|T, P_T|T)
        PT = P_filt[T_obs-1] + 1e-10 * Ik
        L  = np.linalg.cholesky(PT)
        betas[T_obs-1] = beta_filt[T_obs-1] + L @ np.random.randn(k)

        for t in range(T_obs-2, -1, -1):
            Pt  = P_filt[t]
            bft = beta_filt[t]

            # P_{t+1|t} = F Pt F' + Q  (diagonal)
            Pt1p = np.zeros((k, k))
            for i in range(k):
                for j in range(k):
                    Pt1p[i, j] = f_vec[i] * Pt[i, j] * f_vec[j]
            for i in range(k):
                Pt1p[i, i] += q_vec[i]
            Pt1p = 0.5 * (Pt1p + Pt1p.T)

            # Ganancia suavizadora: G_t = Pt F' Pt1p^{-1}
            # Con F diagonal: (Pt F')_{ij} = Pt_{ij} * f_j
            PtFT = np.zeros((k, k))
            for i in range(k):
                for j in range(k):
                    PtFT[i, j] = Pt[i, j] * f_vec[j]
            Gt = np.linalg.solve(Pt1p.T, PtFT.T).T

            # Estado predicho: beta_{t+1|t} = mu + F(beta_t|t - mu)
            b_pred = np.zeros(k)
            for i in range(k):
                b_pred[i] = mu[i] + f_vec[i] * (bft[i] - mu[i])

            # Media condicional: beta_t|t + G_t(beta_{t+1} - beta_{t+1|t})
            mu_s = bft + Gt @ (betas[t+1] - b_pred)

            # Covarianza condicional: Pt - G_t Pt+1|t G_t'
            Ps = Pt - Gt @ Pt1p @ Gt.T
            Ps = 0.5 * (Ps + Ps.T) + 1e-10 * Ik

            L = np.linalg.cholesky(Ps)
            betas[t] = mu_s + L @ np.random.randn(k)

        return betas

else:
    # Version Python puro (sin Numba) — misma logica, mas lenta
    def _ffbs_diag(mu, f_vec, q_vec, r_vec, H, yields):
        T_obs, N_obs = yields.shape
        k = len(mu)
        Ik = np.eye(k)
        beta_filt = np.zeros((T_obs, k))
        P_filt    = np.zeros((T_obs, k, k))

        beta_hat = mu.copy()
        P = np.diag(q_vec / np.maximum(1 - f_vec**2, 1e-6))
        F_mat = np.diag(f_vec)
        Q_mat = np.diag(q_vec)
        R_mat = np.diag(r_vec)

        for t in range(T_obs):
            beta_p = mu + F_mat @ (beta_hat - mu)
            P_p    = F_mat @ P @ F_mat.T + Q_mat
            P_p    = 0.5*(P_p + P_p.T)

            v  = yields[t] - H @ beta_p
            HP = H @ P_p
            S  = HP @ H.T + R_mat
            K  = np.linalg.solve(S, HP).T

            beta_hat = beta_p + K @ v
            IKH = Ik - K @ H
            P   = IKH @ P_p @ IKH.T + K @ R_mat @ K.T
            P   = 0.5*(P + P.T)
            beta_filt[t] = beta_hat; P_filt[t] = P

        betas = np.zeros((T_obs, k))
        PT = P_filt[-1] + 1e-10*Ik
        betas[-1] = beta_filt[-1] + np.linalg.cholesky(PT) @ np.random.randn(k)

        for t in range(T_obs-2, -1, -1):
            Pt  = P_filt[t]
            bft = beta_filt[t]
            Pt1p = F_mat @ Pt @ F_mat.T + Q_mat
            Pt1p = 0.5*(Pt1p + Pt1p.T)
            Gt   = np.linalg.solve(Pt1p.T, (Pt @ F_mat.T).T).T
            b_pred = mu + F_mat @ (bft - mu)
            mu_s = bft + Gt @ (betas[t+1] - b_pred)
            Ps   = Pt - Gt @ Pt1p @ Gt.T
            Ps   = 0.5*(Ps + Ps.T) + 1e-10*Ik
            betas[t] = mu_s + np.linalg.cholesky(Ps) @ np.random.randn(k)

        return betas


def ffbs(mu, f_vec, q_vec, r_vec, H, yields):
    '''Wrapper publico de FFBS para F diagonal.'''
    return _ffbs_diag(
        np.asarray(mu, dtype=np.float64),
        np.asarray(f_vec, dtype=np.float64),
        np.asarray(q_vec, dtype=np.float64),
        np.asarray(r_vec, dtype=np.float64),
        np.asarray(H, dtype=np.float64),
        np.asarray(yields, dtype=np.float64),
    )

print("FFBS definido (modo:", "Numba JIT" if _NUMBA else "Python puro", ")")


## CELDA 4: Log-verosimilitud marginal $p(\mathbf{Y}|\boldsymbol{\tau}, \boldsymbol{\theta})$

Para el paso MH de $\boldsymbol{\tau}$, necesitamos evaluar la log-verosimilitud **con $\beta_{1:T}$
integrado** (marginalizando), que es precisamente la log-verosimilitud del Filtro de Kalman.

Esto es correcto porque comparar propuestas usando una trayectoria fija $\beta_{1:T}$
(que fue muestreada con el $\tau$ anterior) introduce un sesgo: la trayectoria es
inconsistente con el nuevo $\tau^*$. En cambio, la log-lik marginal promedia correctamente
sobre todas las trayectorias posibles.


In [ ]:
if _NUMBA:
    @_njit(cache=True)
    def _kf_loglik_diag(mu, f_vec, q_vec, r_vec, H, yields):
        '''
        Log-verosimilitud marginal p(Y|tau, theta) via KF con F diagonal.
        Valido para k=3 (DNS) y k=4 (DNSS).
        '''
        T_obs, N_obs = yields.shape
        k = len(mu)
        LOG2PI = np.log(2.0 * np.pi)

        beta_hat = mu.copy()
        P = np.zeros((k, k))
        for i in range(k):
            P[i, i] = q_vec[i] / max(1.0 - f_vec[i]**2, 1e-6)
        Ik = np.eye(k)

        log_lik = 0.0
        for t in range(T_obs):
            beta_p = np.zeros(k)
            for i in range(k):
                beta_p[i] = mu[i] + f_vec[i] * (beta_hat[i] - mu[i])

            P_p = np.zeros((k, k))
            for i in range(k):
                for j in range(k):
                    P_p[i, j] = f_vec[i] * P[i, j] * f_vec[j]
            for i in range(k):
                P_p[i, i] += q_vec[i]
            P_p = 0.5 * (P_p + P_p.T)

            v  = yields[t] - H @ beta_p
            HP = H @ P_p
            S  = HP @ H.T
            for n in range(N_obs):
                S[n, n] += r_vec[n]
            S = 0.5 * (S + S.T)

            # log|S| y cuadratica via Cholesky
            L_ch = np.linalg.cholesky(S + 1e-10*np.eye(N_obs))
            log_det = 0.0
            for n in range(N_obs):
                log_det += np.log(L_ch[n, n])
            log_det *= 2.0
            z    = np.linalg.solve(L_ch, v)
            quad = np.dot(z, z)
            log_lik += -0.5 * (N_obs*LOG2PI + log_det + quad)

            K = np.linalg.solve(S, HP).T
            beta_hat = beta_p + K @ v
            IKH = Ik - K @ H
            Rmat = np.zeros((N_obs, N_obs))
            for n in range(N_obs):
                Rmat[n, n] = r_vec[n]
            P = IKH @ P_p @ IKH.T + K @ Rmat @ K.T
            P = 0.5 * (P + P.T)

        return log_lik

else:
    def _kf_loglik_diag(mu, f_vec, q_vec, r_vec, H, yields):
        T_obs, N_obs = yields.shape
        k = len(mu)
        Ik = np.eye(k); LOG2PI = np.log(2*np.pi)
        F_mat = np.diag(f_vec); Q_mat = np.diag(q_vec); R_mat = np.diag(r_vec)
        beta_hat = mu.copy()
        P = np.diag(q_vec / np.maximum(1-f_vec**2, 1e-6))
        log_lik = 0.0
        for t in range(T_obs):
            beta_p = mu + F_mat @ (beta_hat - mu)
            P_p    = F_mat @ P @ F_mat.T + Q_mat; P_p = 0.5*(P_p+P_p.T)
            v = yields[t] - H @ beta_p
            S = H @ P_p @ H.T + R_mat; S = 0.5*(S+S.T)
            L = np.linalg.cholesky(S + 1e-10*np.eye(N_obs))
            log_det = 2.0*np.sum(np.log(np.diag(L)))
            z = np.linalg.solve(L, v); quad = z@z
            log_lik += -0.5*(N_obs*LOG2PI + log_det + quad)
            K = np.linalg.solve(S, H@P_p).T
            beta_hat = beta_p + K @ v
            IKH = Ik - K@H
            P = IKH @ P_p @ IKH.T + K @ R_mat @ K.T; P = 0.5*(P+P.T)
        return log_lik


def kf_loglik(mu, f_vec, q_vec, r_vec, H, yields):
    '''Wrapper publico de log-verosimilitud marginal KF.'''
    return float(_kf_loglik_diag(
        np.asarray(mu,    dtype=np.float64),
        np.asarray(f_vec, dtype=np.float64),
        np.asarray(q_vec, dtype=np.float64),
        np.asarray(r_vec, dtype=np.float64),
        np.asarray(H,     dtype=np.float64),
        np.asarray(yields,dtype=np.float64),
    ))

print("Log-verosimilitud marginal KF definida")


## CELDA 5: Posteriors conjugadas para $F$, $\boldsymbol{\mu}$, $Q$, $R$

Dado el bloque FFBS que muestrea $\boldsymbol{\beta}_{1:T}$, los demás parámetros tienen
**distribuciones posteriores analíticas** (conjugadas), lo que hace que el muestreo
sea directo y exacto.

**Para $F$ diagonal y $\boldsymbol{\mu}$** (Gibbs secuencial por factor $i$):

$$\mu_i | f_i, \boldsymbol{\beta}, q_i \sim \mathcal{N}\!\left(
  \frac{m_{0i}/V_{0i} + (1-f_i)\sum_t(\beta_{i,t} - f_i\beta_{i,t-1})/q_i}
       {1/V_{0i} + (1-f_i)^2 T_\text{eff}/q_i},\;
  \left[1/V_{0i} + (1-f_i)^2 T_\text{eff}/q_i\right]^{-1}
\right)$$

$$f_i | \mu_i, \boldsymbol{\beta}, q_i \sim \mathcal{TN}_{(-1,1)}\!\left(
  \frac{f_{0i}/V_{fi} + \sum_t a_{i,t}a_{i,t-1}/q_i}
       {1/V_{fi} + \sum_t a_{i,t-1}^2/q_i},\;
  \left[1/V_{fi} + \sum_t a_{i,t-1}^2/q_i\right]^{-1}
\right)$$

donde $a_{i,t} = \beta_{i,t} - \mu_i$.

**Para $q_i$ y $r_n$** (Inverse-Gamma conjugada):

$$q_i | \boldsymbol{\beta}, \mu_i, f_i \sim \mathcal{IG}\!\left(a_0 + \frac{T_\text{eff}}{2},\; b_0 + \frac{\sum_t \eta_{i,t}^2}{2}\right)$$

$$r_n | \boldsymbol{\beta}, Y \sim \mathcal{IG}\!\left(a_0 + \frac{T}{2},\; b_0 + \frac{\sum_t \varepsilon_{n,t}^2}{2}\right)$$


In [ ]:
def sample_F_mu_diagonal(betas, f_vec, q_vec,
                           prior_f_mean, prior_f_var,
                           prior_mu_mean, prior_mu_var):
    '''
    Muestrea F (diagonal) y mu de sus posteriors conjugadas.
    Itera secuencialmente: mu_i | f_i → f_i | mu_i para cada factor i.
    Valido para cualquier k (DNS: k=3, DNSS: k=4).
    '''
    k      = len(f_vec)
    f_new  = f_vec.copy()
    mu_new = np.zeros(k)
    bt      = betas[1:, :]    # t=1,...,T-1
    bt_lag  = betas[:-1, :]   # t=0,...,T-2
    T_eff   = bt.shape[0]

    for i in range(k):
        qi = q_vec[i]; fi = f_new[i]

        # ── Posterior de mu_i | f_i ──────────────────────────────────────────
        lhs    = bt[:, i] - fi * bt_lag[:, i]   # beta_{i,t} - fi*beta_{i,t-1}
        Xi     = 1.0 - fi
        prec_m = 1.0/prior_mu_var[i] + Xi**2 * T_eff / qi
        mean_m = (prior_mu_mean[i]/prior_mu_var[i] + Xi/qi * lhs.sum()) / prec_m
        var_m  = 1.0 / prec_m
        mu_new[i] = np.random.normal(mean_m, np.sqrt(max(var_m, 1e-15)))

        # ── Posterior de f_i | mu_i_new ──────────────────────────────────────
        mui    = mu_new[i]
        ai_t   = bt[:, i]    - mui
        ai_lag = bt_lag[:, i] - mui
        s1     = np.sum(ai_lag**2)
        s2     = np.sum(ai_t * ai_lag)
        prec_f = 1.0/prior_f_var[i] + s1/qi
        mean_f = (prior_f_mean[i]/prior_f_var[i] + s2/qi) / prec_f
        var_f  = 1.0 / prec_f; std_f = np.sqrt(max(var_f, 1e-15))

        # Truncada en (-1, 1) para garantizar estacionariedad
        a_tr = (-1.0 - mean_f) / std_f
        b_tr = ( 1.0 - mean_f) / std_f
        f_new[i] = truncnorm.rvs(a_tr, b_tr, loc=mean_f, scale=std_f)

    return f_new, mu_new


def sample_Q(betas, mu, f_vec, prior_a, prior_b):
    '''
    Muestrea Q diagonal de Inverse-Gamma conjugada.
    eta_{i,t} = beta_{i,t} - mu_i - f_i*(beta_{i,t-1} - mu_i)
    '''
    k      = len(f_vec)
    q_new  = np.zeros(k)
    bt     = betas[1:, :]; bt_lag = betas[:-1, :]
    T_eff  = bt.shape[0]
    for i in range(k):
        eta      = bt[:,i] - mu[i] - f_vec[i]*(bt_lag[:,i] - mu[i])
        a_post   = prior_a + T_eff / 2.0
        b_post   = prior_b + np.sum(eta**2) / 2.0
        q_new[i] = invgamma.rvs(a_post, scale=b_post)
    return q_new


def sample_R(yields, betas, H, prior_a, prior_b, obs_weights=None):
    '''
    Muestrea R diagonal de Inverse-Gamma conjugada.
    eps_{n,t} = y_{t,n} - H_n @ beta_t
    obs_weights: (N,) — reduce prior_b para plazos con mayor peso (Opcion C).
    '''
    T_obs, N_obs = yields.shape
    r_new  = np.zeros(N_obs)
    fitted = (H @ betas.T).T   # (T, N)
    resid  = yields - fitted

    for n in range(N_obs):
        sse    = np.sum(resid[:, n]**2)
        a_post = prior_a + T_obs / 2.0
        # Opcion C: peso mayor → prior_b efectivo menor → R_n menor → mejor ajuste
        w_n    = obs_weights[n] if obs_weights is not None else 1.0
        b_post = prior_b/max(w_n, 1e-6) + sse / 2.0
        r_new[n] = invgamma.rvs(a_post, scale=b_post)
    return r_new


def sample_R_groups(yields, betas, H, prior_a, prior_b,
                    maturities, group_thresholds=(3.0, 10.0)):
    '''
    Opcion A: Muestrea R con estructura por grupos (3 parametros en lugar de N).
    corto <= th1, medio <= th2, largo > th2.
    '''
    T_obs, N_obs = yields.shape
    th1, th2 = group_thresholds
    fitted   = (H @ betas.T).T
    resid    = yields - fitted
    r_new    = np.zeros(N_obs)

    for g_idx, (lo, hi) in enumerate([(0, th1), (th1, th2), (th2, np.inf)]):
        mask  = (maturities > lo) & (maturities <= hi)
        if not mask.any(): continue
        sse_g  = np.sum(resid[:, mask]**2)
        ng     = mask.sum()
        a_post = prior_a + T_obs * ng / 2.0
        b_post = prior_b + sse_g / 2.0
        r_g    = invgamma.rvs(a_post, scale=b_post)
        r_new[mask] = r_g

    return r_new

print("Posteriors conjugadas definidas")


## CELDA 6: Metropolis-Hastings para $\boldsymbol{\tau}$

$\boldsymbol{\tau}$ no tiene posterior conjugada, por lo que se usa MH con propuesta
**random walk en log-escala** (garantiza positividad sin truncar):

$$\log\tau_1^* = \log\tau_1 + \sigma_1\,z_1, \quad z_1 \sim \mathcal{N}(0,1)$$

La ratio de aceptación (con prior normal o uniforme):

$$\log\alpha = \underbrace{[\log p(Y|\tau^*, \theta) - \log p(Y|\tau, \theta)]}_{\text{ratio log-lik}}
+ \underbrace{[\log p(\tau^*) - \log p(\tau)]}_{\text{ratio prior}}$$

El Jacobiano del cambio de variables $\log\tau \to \tau$ cancela porque la propuesta
es simétrica en log-escala (random walk gaussiano).

**Tasa de aceptación óptima:** ~44% (1D) y ~23% (2D) según Roberts & Rosenthal (2001).
El MH se adapta durante el burn-in ajustando $\sigma$ para alcanzar estos objetivos.


In [ ]:
def _log_prior_tau(tau1, tau2, prior_tau):
    '''
    Evalua log-prior de tau.
    prior_tau puede ser:
      ('uniforme', lo, hi)         — log-prior = 0 si dentro de [lo,hi]
      ('normal', mu1, sd1, mu2, sd2) — Normal independiente para tau1 y tau2
    '''
    kind = prior_tau[0]
    if kind == 'uniforme':
        lo, hi = prior_tau[1], prior_tau[2]
        in1 = (lo < tau1 < hi)
        in2 = (tau2 is None) or (lo < tau2 < hi)
        return 0.0 if (in1 and in2) else -np.inf
    elif kind == 'normal':
        mu1, sd1 = prior_tau[1], prior_tau[2]
        lp = -0.5 * ((tau1 - mu1)/sd1)**2
        if tau2 is not None:
            mu2, sd2 = prior_tau[3], prior_tau[4]
            lp += -0.5 * ((tau2 - mu2)/sd2)**2
        return lp
    return 0.0


def sample_tau_MH_dns(tau1, yields, mu, f_vec, q_vec, r_vec,
                      maturities, prop_std, tau_bounds, prior_tau):
    '''
    MH 1D para tau1 (DNS).
    Retorna: tau1_new, loglik_new, accepted (bool)
    '''
    lo, hi = tau_bounds

    H_curr      = ns_loadings(maturities, tau1)
    loglik_curr = kf_loglik(mu, f_vec, q_vec, r_vec, H_curr, yields)
    lp_curr     = _log_prior_tau(tau1, None, prior_tau)

    log_tau1_p = np.log(tau1) + prop_std * np.random.randn()
    tau1_p     = np.exp(log_tau1_p)

    if not (lo < tau1_p < hi):
        return tau1, loglik_curr, False

    H_prop      = ns_loadings(maturities, tau1_p)
    loglik_prop = kf_loglik(mu, f_vec, q_vec, r_vec, H_prop, yields)
    lp_prop     = _log_prior_tau(tau1_p, None, prior_tau)

    log_alpha = (loglik_prop - loglik_curr) + (lp_prop - lp_curr)

    if np.isfinite(log_alpha) and np.log(np.random.rand()) < log_alpha:
        return tau1_p, loglik_prop, True
    return tau1, loglik_curr, False


def sample_tau_MH_dnss(tau1, tau2, yields, mu, f_vec, q_vec, r_vec,
                       maturities, prop_std, tau_bounds, prior_tau, sep_min=0.15):
    '''
    MH 2D independiente para (tau1, tau2) (DNSS).
    Propone cada tau por separado en log-escala.
    Retorna: tau1_new, tau2_new, loglik_new, accepted (bool)
    '''
    lo, hi = tau_bounds

    H_curr      = nss_loadings(maturities, tau1, tau2)
    loglik_curr = kf_loglik(mu, f_vec, q_vec, r_vec, H_curr, yields)
    lp_curr     = _log_prior_tau(tau1, tau2, prior_tau)

    # Propuesta conjunta en log-escala
    tau1_p = np.exp(np.log(tau1) + prop_std[0] * np.random.randn())
    tau2_p = np.exp(np.log(tau2) + prop_std[1] * np.random.randn())

    if (not (lo < tau1_p < hi)) or (not (lo < tau2_p < hi)) \
       or (abs(tau1_p - tau2_p) < sep_min):
        return tau1, tau2, loglik_curr, False

    H_prop      = nss_loadings(maturities, tau1_p, tau2_p)
    loglik_prop = kf_loglik(mu, f_vec, q_vec, r_vec, H_prop, yields)
    lp_prop     = _log_prior_tau(tau1_p, tau2_p, prior_tau)

    log_alpha = (loglik_prop - loglik_curr) + (lp_prop - lp_curr)

    if np.isfinite(log_alpha) and np.log(np.random.rand()) < log_alpha:
        return tau1_p, tau2_p, loglik_prop, True
    return tau1, tau2, loglik_curr, False

print("MH para tau definido (DNS: 1D | DNSS: 2D)")


## CELDA 7: Especificación de priors

Los priors siguen Caldeira et al. (2010) y Çakmaklı (2013): **débilmente informativos**,
centrados en valores económicamente plausibles pero lo suficientemente difusos
para que los datos dominen en muestras de tamaño razonable.

| Parámetro | Prior | Justificación |
|---|---|---|
| $f_i$ | $\mathcal{TN}_{(-1,1)}\!(0.90,\,0.10^2)$ | Alta persistencia es típica en DNS |
| $\mu_i$ | $\mathcal{N}(m_{0i},\,25)$ | Difuso, centrado en valores históricos |
| $q_i$, $r_n$ | $\mathcal{IG}(0.01,\,0.01)$ | Casi no informativo (Jeffreys-like) |
| $\tau_1$ | $\mathcal{U}(0.15, 5.0)$ o $\mathcal{N}(1.78, 0.40^2)$ | Bounds económicos o informativo |
| $\tau_2$ | $\mathcal{U}(0.15, 5.0)$ o $\mathcal{N}(0.60, 0.08^2)$ | Solo DNSS |


In [ ]:
def make_priors(model='DNS', prior_tau_type='uniforme',
               tau1_mu=1.78, tau1_sd=0.40,
               tau2_mu=0.60, tau2_sd=0.08,
               tau_lo=0.15,  tau_hi=5.0):
    '''
    Construye el diccionario de priors para el Gibbs sampler.

    Parametros
    ----------
    model          : 'DNS' o 'DNSS'
    prior_tau_type : 'uniforme' — prior plano sobre (tau_lo, tau_hi)
                     'normal'   — prior informativo normal sobre tau
                                  Util cuando hump debe estar en un rango conocido

    Retorna dict con todas las especificaciones de priors.
    '''
    k = 3 if model == 'DNS' else 4

    # F diagonal: persistencia alta, tipica de factores DNS
    prior_f_mean = np.array([0.90, 0.90, 0.90, 0.90])[:k]
    prior_f_var  = np.array([0.10, 0.10, 0.15, 0.15])[:k]**2

    # mu: difuso, centrado en valores historicos plausibles
    prior_mu_mean = np.array([5.0, -2.0, 0.0, 0.0])[:k]  # nivel ~5%, pendiente neg.
    prior_mu_var  = np.full(k, 25.0)                       # muy difuso

    # Q y R: IG(0.01, 0.01) — casi no-informativa
    prior_Q_a = 0.01; prior_Q_b = 0.01
    prior_R_a = 0.01; prior_R_b = 0.01

    # Prior de tau
    if prior_tau_type == 'uniforme':
        prior_tau = ('uniforme', tau_lo, tau_hi)
    else:  # normal
        if model == 'DNS':
            prior_tau = ('normal', tau1_mu, tau1_sd, None, None)
        else:
            prior_tau = ('normal', tau1_mu, tau1_sd, tau2_mu, tau2_sd)

    return {
        'prior_f_mean' : prior_f_mean,
        'prior_f_var'  : prior_f_var,
        'prior_mu_mean': prior_mu_mean,
        'prior_mu_var' : prior_mu_var,
        'prior_Q_a'    : prior_Q_a,
        'prior_Q_b'    : prior_Q_b,
        'prior_R_a'    : prior_R_a,
        'prior_R_b'    : prior_R_b,
        'prior_tau'    : prior_tau,
        'tau_bounds'   : (tau_lo, tau_hi),
        'model'        : model,
        'k'            : k,
    }


# Priors por defecto (para previsualizar)
priors_dns  = make_priors('DNS',  'uniforme')
priors_dnss = make_priors('DNSS', 'uniforme')

print("Priors definidas:")
for k, v in priors_dns.items():
    if k not in ('model','k','tau_bounds','prior_tau'):
        print(f"  {k}: {v}")
print(f"  prior_tau: {priors_dns['prior_tau']}")


## CELDA 8: Inicialización de cadenas y Gibbs sampler (una cadena)

## CELDA 7b: Priors extendidos para τ AR(1)

Para el modo `'ar1'`, τ_t sigue una dinámica en log-escala:
$$x_t = \log\tau_t, \qquad x_t = \mu_x + \phi_x(x_{t-1} - \mu_x) + \sigma_x\,\varepsilon_t, \quad \varepsilon_t \sim \mathcal{N}(0,1)$$

Los priors sobre los hiperparámetros del proceso AR(1) de τ son:
- $\phi_x \sim \mathcal{TN}_{(0,1)}(0.95,\ 0.05^2)$ — alta persistencia esperada
- $\sigma_x \sim \mathcal{IG}(2,\ 0.01)$ — varianza pequeña (τ no cambia abruptamente)
- $\mu_x = \log(\tau_\text{init})$ — fijado en la inicialización OLS


In [ ]:
def make_priors_ar1(model, tau1_init, tau2_init=None,
                    phi_x_mean=0.95, phi_x_sd=0.05,
                    sigma_x_a=2.0,   sigma_x_b=0.01,
                    **kwargs):
    '''
    Extiende make_priors() con hiperparametros para el proceso AR(1) de tau.

    Los hiperparametros del proceso x_t = log(tau_t) son:
      phi_x   ~ TruncNormal(0,1)(phi_x_mean, phi_x_sd^2) — persistencia
      sigma_x ~ IG(sigma_x_a, sigma_x_b) — volatilidad del proceso
      mu_x    = log(tau_init) fijo en inicializacion OLS

    Parametros adicionales
    ----------------------
    tau1_init    : float  valor inicial de tau1 (define mu_x1)
    tau2_init    : float  valor inicial de tau2 (define mu_x2, solo DNSS)
    phi_x_mean   : float  media prior de phi_x (default 0.95)
    phi_x_sd     : float  std prior de phi_x (default 0.05)
    sigma_x_a/b  : float  hiperparametros IG para sigma_x (default 2, 0.01)
    '''
    priors = make_priors(model, **kwargs)
    priors['tau_ar1'] = {
        'mu_x1'      : np.log(tau1_init),
        'mu_x2'      : np.log(tau2_init) if tau2_init else None,
        'phi_x_mean' : phi_x_mean,
        'phi_x_var'  : phi_x_sd**2,
        'sigma_x_a'  : sigma_x_a,
        'sigma_x_b'  : sigma_x_b,
    }
    return priors

print("make_priors_ar1() definido")


## CELDA 8b: Modo AR(1) — τ_t dinámico via FFBS auxiliar

### ¿Por qué un FFBS auxiliar para τ_t?

Cuando τ_t sigue un proceso AR(1) en log-escala, el modelo se vuelve
**no lineal en los estados**: las cargas $H_t = H(\tau_t)$ dependen del estado
latente $x_t = \log\tau_t$ de forma no lineal. El FFBS estándar no aplica
directamente porque requiere linealidad.

**Solución: linealización local** (Kim & Nelson 1999, Nakajima 2011).

En cada iteración Gibbs, dado $\bbet_{1:T}$ y $\theta$ actuales, el residuo
$\tilde{y}_{t,n} = y_{t,n} - \bH_{\bar{\tau}}\bbet_t$ se aproxima como
función lineal de $x_t = \log\tau_t$ en torno al valor actual $\bar{x}_t$:

$$\tilde{y}_{t,n} \approx \frac{\partial H_n}{\partial x}(\bar{x}_t)\,(x_t - \bar{x}_t)
  + \text{const} + \varepsilon_{t,n}$$

Esto define un modelo espacio-estado lineal en $x_t$ que puede resolverse
con FFBS. El proceso se itera hasta convergencia (típicamente 3-5 pasos).

### Alternativa: MH por bloque (más simple, menos eficiente)

Para cada $t$ se propone $x_t^* = x_t + \sigma_{mh}\,z$, $z\sim\mathcal{N}(0,1)$,
y se acepta/rechaza con ratio de verosimilitud local. Es más sencillo de
implementar y converge, aunque con mayor autocorrelación en la cadena.
En este notebook implementamos ambos; el bloque AR(1) usa MH por defecto
con opción de FFBS linealizado.


In [ ]:
# ── Funciones auxiliares para modo AR(1) ─────────────────────────────────────

def _ar1_loglik_tau_t(x_t, betas_t, yields_t, r_vec, model, maturities):
    '''
    Log-verosimilitud local p(y_t | x_t, beta_t, r) para un periodo t.
    x_t = log(tau1_t) [DNS] o [log(tau1_t), log(tau2_t)] [DNSS]
    '''
    if model == 'DNS':
        tau1_t = np.exp(x_t[0])
        H_t    = ns_loadings(maturities, tau1_t)
    else:
        tau1_t = np.exp(x_t[0]); tau2_t = np.exp(x_t[1])
        if abs(tau1_t - tau2_t) < 0.15:
            return -np.inf
        H_t = nss_loadings(maturities, tau1_t, tau2_t)

    resid    = yields_t - H_t @ betas_t
    log_lik  = -0.5 * np.sum(resid**2 / r_vec + np.log(2*np.pi*r_vec))
    return float(log_lik)


def sample_tau_ar1_MH(x_seq, betas, yields, r_vec, model, maturities,
                       phi_x, mu_x, sigma_x, prop_std_t=0.05):
    '''
    Muestrea la trayectoria x_1:T = log(tau_t) via MH individual por periodo.

    Para cada t, propone x_t* = x_t + sigma*z y acepta con ratio:
      log_alpha = [loglik(y_t|x_t*) - loglik(y_t|x_t)]
                + [logp(x_t*|x_{t-1},x_{t+1}) - logp(x_t|x_{t-1},x_{t+1})]

    La prior condicional de x_t dado sus vecinos es:
      p(x_t | x_{t-1}, x_{t+1}) proporcional a exp(-0.5*(eta_t^2 + eta_{t+1}^2)/sigma_x^2)

    Parametros
    ----------
    x_seq     : (T, n_tau)  secuencia actual de log(tau_t)
    betas     : (T, k)      factores del FFBS actual
    yields    : (T, N)
    phi_x     : float o (n_tau,)  persistencia AR(1)
    mu_x      : float o (n_tau,)  media de largo plazo de log(tau)
    sigma_x   : float o (n_tau,)  std de innovaciones AR(1)
    prop_std_t: float  sigma de la propuesta MH por periodo

    Retorna
    -------
    x_seq_new : (T, n_tau)  trayectoria actualizada
    acc_rate  : float       tasa de aceptacion media
    '''
    T    = len(x_seq)
    n_x  = x_seq.shape[1]
    x_new = x_seq.copy()
    n_acc = 0

    phi_arr = np.atleast_1d(phi_x)   * np.ones(n_x)
    mu_arr  = np.atleast_1d(mu_x)    * np.ones(n_x)
    sig_arr = np.atleast_1d(sigma_x) * np.ones(n_x)

    for t in range(T):
        x_curr = x_new[t].copy()
        x_prop = x_curr + prop_std_t * np.random.randn(n_x)

        # Log-verosimilitud local
        ll_curr = _ar1_loglik_tau_t(x_curr, betas[t], yields[t], r_vec, model, maturities)
        ll_prop = _ar1_loglik_tau_t(x_prop, betas[t], yields[t], r_vec, model, maturities)

        if not np.isfinite(ll_prop):
            continue

        # Log-prior condicional (de la dinamica AR(1))
        def _lp_cond(x_t, t_idx):
            lp = 0.0
            for j in range(n_x):
                # Contribucion de eta_t = x_t - (mu + phi*(x_{t-1} - mu))
                if t_idx > 0:
                    eta_t = x_t[j] - (mu_arr[j] + phi_arr[j]*(x_new[t_idx-1,j] - mu_arr[j]))
                    lp += -0.5 * eta_t**2 / sig_arr[j]**2
                # Contribucion de eta_{t+1} = x_{t+1} - (mu + phi*(x_t - mu))
                if t_idx < T-1:
                    eta_t1 = x_new[t_idx+1,j] - (mu_arr[j] + phi_arr[j]*(x_t[j] - mu_arr[j]))
                    lp += -0.5 * eta_t1**2 / sig_arr[j]**2
            return lp

        lp_curr = _lp_cond(x_curr, t)
        lp_prop = _lp_cond(x_prop, t)

        log_alpha = (ll_prop - ll_curr) + (lp_prop - lp_curr)
        if np.isfinite(log_alpha) and np.log(np.random.rand()) < log_alpha:
            x_new[t] = x_prop
            n_acc    += 1

    return x_new, n_acc / T


def sample_phi_x(x_seq, mu_x, sigma_x, phi_x_mean, phi_x_var):
    '''
    Muestrea phi_x de Normal truncada conjugada dado x_1:T.
    Aplica para cada componente de tau independientemente.
    '''
    n_x    = x_seq.shape[1]
    phi_new = np.zeros(n_x)
    for j in range(n_x):
        a_t   = x_seq[1:, j]   - mu_x[j]   # x_t - mu_x
        a_lag = x_seq[:-1, j]  - mu_x[j]   # x_{t-1} - mu_x
        s2    = sigma_x[j]**2
        prec  = 1.0/phi_x_var + np.sum(a_lag**2) / s2
        mean  = (phi_x_mean/phi_x_var + np.sum(a_t*a_lag)/s2) / prec
        std   = 1.0/np.sqrt(prec)
        # Truncada en (0, 1) — phi_x > 0 garantiza tau > 0
        a_tr, b_tr = (0.0 - mean)/std, (1.0 - mean)/std
        phi_new[j] = truncnorm.rvs(a_tr, b_tr, loc=mean, scale=std)
    return phi_new


def sample_sigma_x(x_seq, mu_x, phi_x, sigma_x_a, sigma_x_b):
    '''
    Muestrea sigma_x de Inverse-Gamma conjugada dado x_1:T, phi_x.
    '''
    T_eff  = len(x_seq) - 1
    n_x    = x_seq.shape[1]
    sig_new = np.zeros(n_x)
    for j in range(n_x):
        eta = x_seq[1:,j] - mu_x[j] - phi_x[j]*(x_seq[:-1,j] - mu_x[j])
        a_post = sigma_x_a + T_eff / 2.0
        b_post = sigma_x_b + np.sum(eta**2) / 2.0
        sig_new[j] = np.sqrt(invgamma.rvs(a_post, scale=b_post))
    return sig_new


def _tau_seq_from_x(x_seq):
    '''Convierte secuencia x_t = log(tau_t) a tau_t.'''
    return np.exp(x_seq)   # (T, n_tau)


print("Funciones AR(1) para tau definidas")


## CELDA 8c: Modo Ventana — τ cambia entre ventanas rodantes

### Concepto

En lugar de un único τ global para toda la muestra (modo estático),
se divide la muestra en ventanas y se corre el Gibbs sampler por separado
en cada ventana. Cada ventana tiene su propio τ posterior.

**Ventajas:** τ_t captura cambios de régimen en la curva. Computacionalmente
es $W$ corridas del sampler estático — sin modificar el núcleo del algoritmo.

**Desventajas:** cada ventana tiene mucho menos información que la muestra completa,
lo que amplía los intervalos de credibilidad de τ.

```
ventanas solapadas:  [0, W-1], [step, W-1+step], [2*step, ...]
tau por ventana:     tau_1     tau_2              tau_3  ...
```

La función `run_rolling_bayes()` orquesta las corridas y ensambla los resultados.


In [ ]:
def run_rolling_bayes(yields, maturities, dates, priors,
                      n_iter, n_burnin, window_size, step=None,
                      n_chains=2, verbose=True, **kwargs):
    '''
    Gibbs sampler en ventanas rodantes: tau cambia entre ventanas.

    Divide la muestra en ventanas solapadas de tamano window_size,
    corre run_chains_parallel() en cada una y ensambla los resultados.

    Parametros
    ----------
    window_size : int   tamano de cada ventana (periodos)
    step        : int   desplazamiento entre ventanas (None -> window_size//2)
    **kwargs    : pasan a run_chains_parallel (obs_weights, r_mode, etc.)

    Retorna
    -------
    list de dicts, uno por ventana, con claves adicionales:
      't_start', 't_end', 'dates_window', 'tau1_pm', 'tau2_pm'
    '''
    T = len(yields)
    if step is None:
        step = max(1, window_size // 2)
    model = priors['model']

    starts = list(range(0, T - window_size + 1, step))
    if verbose:
        print(f"Modo ventana: {len(starts)} ventanas | W={window_size} | step={step}")

    resultados = []
    for i, t0 in enumerate(starts):
        t1     = t0 + window_size
        y_win  = yields[t0:t1]
        d_win  = dates[t0:t1]

        if verbose:
            print(f"  Ventana {i+1}/{len(starts)}: t=[{t0},{t1})  "
                  f"{str(d_win[0])[:10]}→{str(d_win[-1])[:10]}")

        chains_w = run_chains_parallel(
            y_win, maturities, priors, n_iter, n_burnin,
            n_chains=n_chains, verbose=False, **kwargs)

        # Resumen por ventana
        tau1_samp = np.concatenate([ch['tau1'] for ch in chains_w])
        tau2_samp = (np.concatenate([ch['tau2'] for ch in chains_w])
                     if model == 'DNSS' else None)

        win_res = {
            'chains'     : chains_w,
            't_start'    : t0,
            't_end'      : t1,
            'dates_window': d_win,
            'tau1_pm'    : float(tau1_samp.mean()),
            'tau1_q025'  : float(np.percentile(tau1_samp, 2.5)),
            'tau1_q975'  : float(np.percentile(tau1_samp, 97.5)),
            'tau2_pm'    : float(tau2_samp.mean()) if tau2_samp is not None else None,
            'tau2_q025'  : float(np.percentile(tau2_samp, 2.5)) if tau2_samp is not None else None,
            'tau2_q975'  : float(np.percentile(tau2_samp, 97.5)) if tau2_samp is not None else None,
        }
        resultados.append(win_res)

    if verbose:
        print(f"\nResumen tau1 por ventana:")
        for r in resultados:
            print(f"  [{r['t_start']:3d},{r['t_end']:3d})  "
                  f"tau1={r['tau1_pm']:.3f} "
                  f"IC95%=[{r['tau1_q025']:.3f},{r['tau1_q975']:.3f}]")
    return resultados


def assemble_rolling_tau(rolling_results, T, step=None):
    '''
    Convierte resultados de ventanas en secuencias tau_t de longitud T.
    Usa la media de las ventanas que cubren cada periodo t.

    Retorna
    -------
    tau1_t : (T,)  media posterior de tau1 en cada periodo
    tau1_lo: (T,)  percentil 2.5%
    tau1_hi: (T,)  percentil 97.5%
    '''
    tau1_sum  = np.zeros(T); tau1_lo_sum = np.zeros(T)
    tau1_hi_sum = np.zeros(T); count = np.zeros(T)
    tau2_sum = np.zeros(T); tau2_lo_sum = np.zeros(T); tau2_hi_sum = np.zeros(T)

    for r in rolling_results:
        t0, t1 = r['t_start'], r['t_end']
        tau1_sum[t0:t1]    += r['tau1_pm']
        tau1_lo_sum[t0:t1] += r['tau1_q025']
        tau1_hi_sum[t0:t1] += r['tau1_q975']
        count[t0:t1]       += 1
        if r['tau2_pm'] is not None:
            tau2_sum[t0:t1]    += r['tau2_pm']
            tau2_lo_sum[t0:t1] += r['tau2_q025']
            tau2_hi_sum[t0:t1] += r['tau2_q975']

    count = np.maximum(count, 1)
    tau1_t  = tau1_sum  / count
    tau1_lo = tau1_lo_sum / count
    tau1_hi = tau1_hi_sum / count
    tau2_t  = (tau2_sum / count) if rolling_results[0]['tau2_pm'] is not None else None
    tau2_lo = (tau2_lo_sum / count) if tau2_t is not None else None
    tau2_hi = (tau2_hi_sum / count) if tau2_t is not None else None
    return tau1_t, tau1_lo, tau1_hi, tau2_t, tau2_lo, tau2_hi

print("Modo ventana definido")


## CELDA 8d: `run_chain()` extendido — soporta los 3 modos de τ

La función `run_chain()` original se reemplaza aquí por la versión v2 que acepta
`tau_mode` y despacha al bloque correspondiente en cada iteración.


In [ ]:
def run_chain(yields, maturities, priors, n_iter, n_burnin, seed,
              tau1_init=None, tau2_init=None,
              prop_std_init=None, adapt_mh=True,
              obs_weights=None, r_mode='libre',
              group_thresholds=(3.0, 10.0),
              tau_mode='estatico',
              tau_window_size=None,
              tau_ar1_prop_std=0.05,
              verbose=False):
    '''
    Gibbs sampler DNS/DNSS — soporta 3 modos de modelado de tau.

    Parametros de tau
    -----------------
    tau_mode : str
        'estatico'  — tau global, posterior via MH (comportamiento original v1)
        'ventana'   — tau cambia cada tau_window_size periodos via sub-sampler
                      (equivalente a run_rolling_bayes pero integrado en la cadena)
        'ar1'       — tau_t sigue AR(1) latente en log-escala, muestreado por
                      MH periodo a periodo con prior AR(1)

    tau_window_size : int (solo para tau_mode='ventana')
        Tamano de ventana. None -> T//4

    tau_ar1_prop_std : float (solo para tau_mode='ar1')
        Std de la propuesta MH por periodo para log(tau_t)

    El dict de retorno incluye:
      'tau1'       : (n_store,)     modo estatico
      'tau1_seq'   : (n_store, T)   modo ar1 — trayectoria completa en cada iteracion
      'phi_x1'     : (n_store,)     modo ar1 — persistencia AR(1) de log(tau1)
      'sigma_x1'   : (n_store,)     modo ar1 — std de innovaciones AR(1) de log(tau1)
    '''
    np.random.seed(seed)
    model    = priors['model']
    k        = priors['k']
    T_obs, N = yields.shape
    n_tau    = 1 if model == 'DNS' else 2   # numero de parametros tau

    # Inicializacion OLS
    if tau1_init is None:
        tau1_init, tau2_init = ols_init_tau(yields, maturities, model)
    elif tau2_init is None and model == 'DNSS':
        _, tau2_init = ols_init_tau(yields, maturities, model)

    state  = initialize_chain(yields, maturities, model, tau1_init,
                              tau2_init if model=='DNSS' else 1.0, seed)
    mu     = state['mu'].copy()
    f_vec  = state['f_vec'].copy()
    q_vec  = state['q_vec'].copy()
    r_vec  = state['r_vec'].copy()
    tau1   = state['tau1']
    tau2   = state['tau2'] if model=='DNSS' else None
    betas  = state['betas'].copy()

    # ── Inicializacion especifica por tau_mode ────────────────────────────────
    if tau_mode == 'ar1':
        ar1_cfg = priors.get('tau_ar1', {})
        mu_x1   = ar1_cfg.get('mu_x1',   np.log(tau1))
        phi_x1  = ar1_cfg.get('phi_x_mean', 0.95)
        sig_x1  = np.sqrt(ar1_cfg.get('sigma_x_b', 0.01) /
                           max(ar1_cfg.get('sigma_x_a', 2.0) - 1, 0.1))
        # Secuencia inicial: constante en tau_init
        x1_seq = np.full((T_obs, 1), np.log(tau1))
        if model == 'DNSS':
            mu_x2  = ar1_cfg.get('mu_x2', np.log(tau2))
            phi_x2 = phi_x1; sig_x2 = sig_x1
            x_seq  = np.column_stack([x1_seq, np.full((T_obs,1), np.log(tau2))])
        else:
            x_seq  = x1_seq
        tau_ar1_info = {
            'mu_x'   : np.array([mu_x1] + ([mu_x2] if model=='DNSS' else [])),
            'phi_x'  : np.array([phi_x1] * n_tau),
            'sigma_x': np.array([sig_x1] * n_tau),
        }
        phi_x_mean = ar1_cfg.get('phi_x_mean', 0.95)
        phi_x_var  = ar1_cfg.get('phi_x_var',  0.05**2)
        sigma_x_a  = ar1_cfg.get('sigma_x_a',  2.0)
        sigma_x_b  = ar1_cfg.get('sigma_x_b',  0.01)

    elif tau_mode == 'ventana':
        win_size = tau_window_size if tau_window_size else max(30, T_obs // 4)
        win_tau1 = tau1
        win_tau2 = tau2

    # ── MH std para modo estatico ─────────────────────────────────────────────
    if prop_std_init is None:
        prop_std = np.array([0.08, 0.06]) if model=='DNSS' else np.array([0.08])
    else:
        prop_std = np.atleast_1d(prop_std_init).copy()

    # ── Storage ───────────────────────────────────────────────────────────────
    n_store = n_iter - n_burnin
    store = {
        'mu'   : np.zeros((n_store, k)),
        'f_vec': np.zeros((n_store, k)),
        'q_vec': np.zeros((n_store, k)),
        'r_vec': np.zeros((n_store, N)),
        'tau1' : np.zeros(n_store),
        'betas': np.zeros((n_store, T_obs, k)),
    }
    if model == 'DNSS':
        store['tau2'] = np.zeros(n_store)
    if tau_mode == 'ar1':
        store['tau1_seq'] = np.zeros((n_store, T_obs))
        store['phi_x1']   = np.zeros(n_store)
        store['sigma_x1'] = np.zeros(n_store)
        if model == 'DNSS':
            store['tau2_seq'] = np.zeros((n_store, T_obs))
            store['phi_x2']   = np.zeros(n_store)
            store['sigma_x2'] = np.zeros(n_store)

    n_mh_acc = 0; n_mh_tot = 0
    adapt_win_size = max(1, n_burnin // 5)
    t0_time = time.time()

    for s in range(n_iter):
        # ── Determinar H para este paso ───────────────────────────────────────
        if tau_mode == 'ar1':
            # H_t variable: usar media de la secuencia actual para FFBS
            # (se actualiza despues del muestreo de betas)
            tau1_t_arr = np.exp(x_seq[:, 0])
            tau2_t_arr = np.exp(x_seq[:, 1]) if model=='DNSS' else None
            # Para FFBS usamos H con tau medio (aproximacion)
            tau1_med = float(np.median(tau1_t_arr))
            tau2_med = float(np.median(tau2_t_arr)) if tau2_t_arr is not None else None
        elif tau_mode == 'ventana':
            # Actualizar tau de ventana cada win_size iteraciones del burn-in
            # En post burn-in, tau se actualiza cada win_size/2 iteraciones
            update_freq = win_size if s < n_burnin else win_size // 2
            if s % max(1, update_freq) == 0:
                # Mini-sampler en ventana local
                t_center = min(s % T_obs, T_obs - win_size)
                t_center = max(0, t_center)
                y_w = yields[t_center:t_center + win_size]
                # Correr un par de iteraciones de MH en la ventana
                for _ in range(5):
                    if model == 'DNS':
                        win_tau1, _, _ = sample_tau_MH_dns(
                            win_tau1, y_w, mu, f_vec, q_vec, r_vec,
                            maturities, prop_std[0],
                            priors['tau_bounds'], priors['prior_tau'])
                    else:
                        win_tau1, win_tau2, _, _ = sample_tau_MH_dnss(
                            win_tau1, win_tau2, y_w, mu, f_vec, q_vec, r_vec,
                            maturities, prop_std,
                            priors['tau_bounds'], priors['prior_tau'])
            tau1 = win_tau1; tau2 = win_tau2
            tau1_med = tau1; tau2_med = tau2
        else:
            tau1_med = tau1; tau2_med = tau2

        H = get_H(maturities, tau1_med,
                  tau2_med if model=='DNSS' else None, model)

        # ── Bloque 1: FFBS ────────────────────────────────────────────────────
        betas = ffbs(mu, f_vec, q_vec, r_vec, H, yields)

        # ── Bloque 2: F, mu ───────────────────────────────────────────────────
        f_vec, mu = sample_F_mu_diagonal(
            betas, f_vec, q_vec,
            priors['prior_f_mean'], priors['prior_f_var'],
            priors['prior_mu_mean'], priors['prior_mu_var'])

        # ── Bloque 3: Q ───────────────────────────────────────────────────────
        q_vec = sample_Q(betas, mu, f_vec, priors['prior_Q_a'], priors['prior_Q_b'])

        # ── Bloque 4: R ───────────────────────────────────────────────────────
        if r_mode == 'grupos':
            r_vec = sample_R_groups(yields, betas, H, priors['prior_R_a'],
                                    priors['prior_R_b'], maturities, group_thresholds)
        else:
            r_vec = sample_R(yields, betas, H, priors['prior_R_a'],
                             priors['prior_R_b'], obs_weights)

        # ── Bloque 5: tau (segun modo) ────────────────────────────────────────
        if tau_mode == 'estatico':
            if model == 'DNS':
                tau1, _, accepted = sample_tau_MH_dns(
                    tau1, yields, mu, f_vec, q_vec, r_vec,
                    maturities, prop_std[0], priors['tau_bounds'], priors['prior_tau'])
            else:
                tau1, tau2, _, accepted = sample_tau_MH_dnss(
                    tau1, tau2, yields, mu, f_vec, q_vec, r_vec,
                    maturities, prop_std, priors['tau_bounds'], priors['prior_tau'])
            n_mh_tot += 1; n_mh_acc += int(accepted)

        elif tau_mode == 'ar1':
            # Muestrear x_t = log(tau_t) via MH por periodo
            x_seq, acc_t = sample_tau_ar1_MH(
                x_seq, betas, yields, r_vec, model, maturities,
                tau_ar1_info['phi_x'], tau_ar1_info['mu_x'],
                tau_ar1_info['sigma_x'], tau_ar1_prop_std)
            n_mh_acc += acc_t; n_mh_tot += 1
            accepted = acc_t > 0

            # Muestrear hiperparametros del proceso AR(1)
            tau_ar1_info['phi_x'] = sample_phi_x(
                x_seq, tau_ar1_info['mu_x'], tau_ar1_info['sigma_x'],
                phi_x_mean, phi_x_var)
            tau_ar1_info['sigma_x'] = sample_sigma_x(
                x_seq, tau_ar1_info['mu_x'], tau_ar1_info['phi_x'],
                sigma_x_a, sigma_x_b)

            # Usar media de la trayectoria como "tau representativo"
            tau1 = float(np.exp(x_seq[:, 0]).mean())
            if model == 'DNSS':
                tau2 = float(np.exp(x_seq[:, 1]).mean())

        elif tau_mode == 'ventana':
            # Ya se actualizó arriba; adaptacion de prop_std
            accepted = True

        # ── Adaptacion MH ─────────────────────────────────────────────────────
        if adapt_mh and tau_mode == 'estatico' and s < n_burnin \
           and (s+1) % adapt_win_size == 0:
            rate   = n_mh_acc / max(n_mh_tot, 1)
            target = 0.40 if model=='DNS' else 0.28
            if   rate < target - 0.12: prop_std *= 0.70
            elif rate > target + 0.15: prop_std *= 1.35
            prop_std  = np.clip(prop_std, 0.005, 0.60)
            n_mh_acc  = 0; n_mh_tot = 0

        # ── Guardar post burn-in ───────────────────────────────────────────────
        if s >= n_burnin:
            idx = s - n_burnin
            store['mu'][idx]    = mu
            store['f_vec'][idx] = f_vec
            store['q_vec'][idx] = q_vec
            store['r_vec'][idx] = r_vec
            store['tau1'][idx]  = tau1
            store['betas'][idx] = betas
            if model == 'DNSS':
                store['tau2'][idx] = tau2
            if tau_mode == 'ar1':
                store['tau1_seq'][idx] = np.exp(x_seq[:, 0])
                store['phi_x1'][idx]   = tau_ar1_info['phi_x'][0]
                store['sigma_x1'][idx] = tau_ar1_info['sigma_x'][0]
                if model == 'DNSS':
                    store['tau2_seq'][idx] = np.exp(x_seq[:, 1])
                    store['phi_x2'][idx]   = tau_ar1_info['phi_x'][1]
                    store['sigma_x2'][idx] = tau_ar1_info['sigma_x'][1]

        if verbose and (s+1) % max(1, n_iter//8) == 0:
            elapsed = time.time() - t0_time
            eta     = (n_iter-s-1) / max((s+1)/elapsed, 1e-6)
            rate_s  = n_mh_acc / max(n_mh_tot, 1) * 100
            tau_str = (f"tau1={tau1:.3f}" if model=='DNS'
                       else f"tau1={tau1:.3f} tau2={tau2:.3f}")
            print(f"  [{s+1:4d}/{n_iter}] {tau_str}  "
                  f"MH:{rate_s:.0f}%  ETA:{eta:.0f}s")

    store['mh_final_rate']  = n_mh_acc / max(n_mh_tot, 1)
    store['prop_std_final'] = prop_std.copy()
    store['model']          = model
    store['priors']         = priors
    store['tau_mode']       = tau_mode
    return store

print("run_chain() v2 definido (soporta tau_mode: estatico | ventana | ar1)")


In [ ]:
def run_chains_parallel(yields, maturities, priors, n_iter, n_burnin,
                         n_chains=4, prop_std_init=None,
                         obs_weights=None, r_mode='libre',
                         group_thresholds=(3.0, 10.0),
                         tau_mode='estatico',
                         tau_window_size=None,
                         tau_ar1_prop_std=0.05,
                         n_jobs=-1, verbose=True):
    '''
    Lanza n_chains cadenas en paralelo con soporte para tau_mode.

    Parametros de tau (nuevos)
    --------------------------
    tau_mode         : 'estatico' | 'ventana' | 'ar1'
    tau_window_size  : solo para 'ventana' — tamano de ventana
    tau_ar1_prop_std : solo para 'ar1' — std de propuesta MH por periodo
    '''
    model = priors['model']
    tau1_base, tau2_base = ols_init_tau(yields, maturities, model)
    pertubs  = np.linspace(-0.10, 0.10, n_chains)
    t1_inits = [tau1_base * np.exp(p) for p in pertubs]
    t2_inits = [(tau2_base * np.exp(p*0.8)) if model=='DNSS' else None
                 for p in pertubs]

    if verbose:
        print(f"Gibbs sampler {model} [{tau_mode}] — {n_chains} cadenas")
        print(f"  n_iter={n_iter} | n_burnin={n_burnin} | "
              f"muestras/cadena={n_iter-n_burnin}")
        if tau_mode == 'ventana':
            wz = tau_window_size or max(30, len(yields)//4)
            print(f"  ventana tau: W={wz}")
        elif tau_mode == 'ar1':
            print(f"  tau AR(1): prop_std_t={tau_ar1_prop_std}")
    t0 = time.time()

    chains = Parallel(n_jobs=min(n_chains, n_jobs) if n_jobs != -1 else n_chains)(
        delayed(run_chain)(
            yields, maturities, priors, n_iter, n_burnin,
            seed=42+j,
            tau1_init=t1_inits[j], tau2_init=t2_inits[j],
            prop_std_init=prop_std_init, adapt_mh=True,
            obs_weights=obs_weights, r_mode=r_mode,
            group_thresholds=group_thresholds,
            tau_mode=tau_mode,
            tau_window_size=tau_window_size,
            tau_ar1_prop_std=tau_ar1_prop_std,
            verbose=False,
        )
        for j in range(n_chains)
    )

    elapsed = time.time() - t0
    if verbose:
        print(f"\nCompletado en {elapsed:.1f}s")
        for j, ch in enumerate(chains):
            rate_str = f"{ch['mh_final_rate']*100:.1f}%"
            print(f"  Cadena {j+1}: MH={rate_str}  tau1~{ch['tau1'].mean():.3f}")
    return chains

print("run_chains_parallel() v2 definido")


In [ ]:
def initialize_chain(yields, maturities, model, tau1_init, tau2_init, seed):
    '''
    Inicializa via OLS two-step (Diebold-Li).
    Proporciona un punto de partida razonable lejos del prior.
    '''
    np.random.seed(seed)
    H  = get_H(maturities, tau1_init, tau2_init, model)
    k  = H.shape[1]
    B  = np.linalg.lstsq(H, yields.T, rcond=None)[0].T  # (T, k)

    mu0   = B.mean(axis=0)
    b_lag = B[:-1]; b_curr = B[1:]
    f0    = np.clip(
        [np.corrcoef(b_lag[:, i], b_curr[:, i])[0, 1] for i in range(k)],
        -0.95, 0.95
    )
    f0 = np.array(f0)

    eta   = b_curr - mu0 - f0*(b_lag - mu0)
    q0    = np.var(eta, axis=0, ddof=1).clip(1e-6)
    eps   = yields - (H @ B.T).T
    r0    = np.var(eps, axis=0, ddof=1).clip(1e-6)

    return {'mu':mu0, 'f_vec':f0, 'q_vec':q0, 'r_vec':r0,
            'tau1':tau1_init, 'tau2':tau2_init, 'betas':B.copy()}


def run_chain(yields, maturities, priors, n_iter, n_burnin, seed,
              tau1_init=None, tau2_init=None,
              prop_std_init=None, adapt_mh=True,
              obs_weights=None, r_mode='libre',
              group_thresholds=(3.0, 10.0),
              verbose=False):
    '''
    Gibbs sampler completo para DNS o DNSS bayesiano.

    Orden de bloques (por iteracion s):
      1. beta_{1:T} | tau, theta  → FFBS
      2. F, mu      | beta, Q     → Normal conjugada (diagonal)
      3. Q          | beta, F, mu → Inverse-Gamma conjugada
      4. R          | beta, Y     → IG conjugada (libre) o grupos (Opcion A)
      5. tau        | theta, Y    → MH 1D (DNS) o 2D (DNSS)

    Parametros nuevos
    -----------------
    obs_weights  : (N,) Opcion C — plazos con mayor peso en posterior de R
    r_mode       : 'libre' → N params en R | 'grupos' → 3 params (Opcion A)
    '''
    np.random.seed(seed)
    model    = priors['model']
    k        = priors['k']
    T_obs, N = yields.shape

    # Tau inicial via OLS si no se provee
    if tau1_init is None:
        tau1_init, tau2_init = ols_init_tau(yields, maturities, model)
    elif tau2_init is None and model == 'DNSS':
        _, tau2_init = ols_init_tau(yields, maturities, model)

    state  = initialize_chain(yields, maturities, model, tau1_init,
                              tau2_init if model=='DNSS' else 1.0, seed)
    mu     = state['mu'].copy()
    f_vec  = state['f_vec'].copy()
    q_vec  = state['q_vec'].copy()
    r_vec  = state['r_vec'].copy()
    tau1   = state['tau1']
    tau2   = state['tau2'] if model=='DNSS' else None
    betas  = state['betas'].copy()

    # Std inicial del MH para tau
    if prop_std_init is None:
        prop_std = np.array([0.08, 0.06]) if model=='DNSS' else np.array([0.08])
    else:
        prop_std = np.atleast_1d(prop_std_init).copy()

    n_store = n_iter - n_burnin
    store   = {
        'mu'   : np.zeros((n_store, k)),
        'f_vec': np.zeros((n_store, k)),
        'q_vec': np.zeros((n_store, k)),
        'r_vec': np.zeros((n_store, N)),
        'tau1' : np.zeros(n_store),
        'betas': np.zeros((n_store, T_obs, k)),
    }
    if model == 'DNSS':
        store['tau2'] = np.zeros(n_store)

    n_mh_acc = 0; n_mh_tot = 0
    adapt_win = max(1, n_burnin // 5)
    t0 = time.time()

    for s in range(n_iter):
        H = get_H(maturities, tau1, tau2, model)

        # Bloque 1: trayectoria completa de beta via FFBS
        betas = ffbs(mu, f_vec, q_vec, r_vec, H, yields)

        # Bloque 2: F diagonal y mu — posteriors conjugadas
        f_vec, mu = sample_F_mu_diagonal(
            betas, f_vec, q_vec,
            priors['prior_f_mean'], priors['prior_f_var'],
            priors['prior_mu_mean'], priors['prior_mu_var'])

        # Bloque 3: Q — Inverse-Gamma conjugada
        q_vec = sample_Q(betas, mu, f_vec, priors['prior_Q_a'], priors['prior_Q_b'])

        # Bloque 4: R — conjugada (libre o grupos segun r_mode)
        if r_mode == 'grupos':
            r_vec = sample_R_groups(yields, betas, H, priors['prior_R_a'],
                                    priors['prior_R_b'], maturities, group_thresholds)
        else:
            r_vec = sample_R(yields, betas, H, priors['prior_R_a'],
                             priors['prior_R_b'], obs_weights)

        # Bloque 5: tau — MH
        if model == 'DNS':
            tau1, _, accepted = sample_tau_MH_dns(
                tau1, yields, mu, f_vec, q_vec, r_vec,
                maturities, prop_std[0], priors['tau_bounds'], priors['prior_tau'])
        else:
            tau1, tau2, _, accepted = sample_tau_MH_dnss(
                tau1, tau2, yields, mu, f_vec, q_vec, r_vec,
                maturities, prop_std, priors['tau_bounds'], priors['prior_tau'])

        n_mh_tot += 1; n_mh_acc += int(accepted)

        # Adaptacion del MH durante burn-in (target: 44% para 1D, 25% para 2D)
        if adapt_mh and s < n_burnin and (s+1) % adapt_win == 0:
            rate   = n_mh_acc / max(n_mh_tot, 1)
            target = 0.40 if model=='DNS' else 0.28
            if   rate < target - 0.12: prop_std *= 0.70
            elif rate > target + 0.15: prop_std *= 1.35
            prop_std = np.clip(prop_std, 0.005, 0.60)
            n_mh_acc = 0; n_mh_tot = 0   # reset ventana

        # Guardar post burn-in
        if s >= n_burnin:
            idx = s - n_burnin
            store['mu'][idx]    = mu
            store['f_vec'][idx] = f_vec
            store['q_vec'][idx] = q_vec
            store['r_vec'][idx] = r_vec
            store['tau1'][idx]  = tau1
            store['betas'][idx] = betas
            if model == 'DNSS':
                store['tau2'][idx] = tau2

        if verbose and (s+1) % max(1, n_iter//10) == 0:
            elapsed = time.time() - t0
            eta     = (n_iter - s - 1) / max((s+1)/elapsed, 1e-6)
            rate_mh = n_mh_acc / max(n_mh_tot, 1) * 100
            tau_str = f"tau1={tau1:.3f}" if model=='DNS' \
                      else f"tau1={tau1:.3f} tau2={tau2:.3f}"
            print(f"  [{s+1:4d}/{n_iter}] MH:{rate_mh:.0f}%  "
                  f"{tau_str}  f=({f_vec[0]:.3f},...) ETA:{eta:.0f}s")

    store['mh_final_rate'] = n_mh_acc / max(n_mh_tot, 1)
    store['prop_std_final'] = prop_std.copy()
    store['model']  = model
    store['priors'] = priors
    return store

print("run_chain() definido")


## CELDA 9: Multi-cadena paralela y diagnósticos de convergencia

Lanzar múltiples cadenas con puntos de inicio distintos es esencial para:
- Detectar multimodalidad en la posterior
- Calcular el estadístico $\hat{R}$ de Gelman-Rubin (requiere $\geq 2$ cadenas)
- Aumentar el ESS efectivo total

**$\hat{R}$ de Gelman-Rubin (1992):**
$$\hat{R} = \sqrt{\frac{\hat{\text{Var}}^+(\psi|Y)}{W}}, \quad
\hat{\text{Var}}^+ = \frac{n-1}{n}W + \frac{B}{n}$$
donde $W$ es la varianza **dentro** de cadenas y $B$ la varianza **entre** cadenas.

| $\hat{R}$ | Interpretación |
|---|---|
| $< 1.05$ | Excelente convergencia |
| $< 1.10$ | Aceptable |
| $\geq 1.10$ | No convergió — extender iteraciones |

**ESS (Effective Sample Size):**
$$\text{ESS} = \frac{n}{1 + 2\sum_{k=1}^\infty \hat{\rho}_k}$$


In [ ]:
def run_chains_parallel(yields, maturities, priors, n_iter, n_burnin,
                         n_chains=4, prop_std_init=None,
                         obs_weights=None, r_mode='libre',
                         group_thresholds=(3.0, 10.0),
                         n_jobs=-1, verbose=True):
    '''
    Lanza n_chains cadenas en paralelo (joblib).
    Cada cadena usa tau_init ligeramente perturbado para explorar el espacio.
    '''
    model = priors['model']
    tau1_base, tau2_base = ols_init_tau(yields, maturities, model)

    # Perturbaciones para diversidad de puntos de inicio
    pertubs = np.linspace(-0.10, 0.10, n_chains)
    t1_inits = [tau1_base * np.exp(p)       for p in pertubs]
    t2_inits = [(tau2_base * np.exp(p*0.8)) if model=='DNSS' else None
                 for p in pertubs]

    if verbose:
        print(f"Gibbs sampler {model} — {n_chains} cadenas paralelas")
        print(f"  n_iter={n_iter} | n_burnin={n_burnin} | "
              f"muestras/cadena={n_iter-n_burnin}")
        print(f"  r_mode='{r_mode}'  obs_weights={'si' if obs_weights is not None else 'no'}")
    t0 = time.time()

    chains = Parallel(n_jobs=min(n_chains, n_jobs) if n_jobs != -1 else n_chains)(
        delayed(run_chain)(
            yields, maturities, priors, n_iter, n_burnin,
            seed=42+j,
            tau1_init=t1_inits[j],
            tau2_init=t2_inits[j],
            prop_std_init=prop_std_init,
            adapt_mh=True,
            obs_weights=obs_weights,
            r_mode=r_mode,
            group_thresholds=group_thresholds,
            verbose=False,
        )
        for j in range(n_chains)
    )

    elapsed = time.time() - t0
    if verbose:
        print(f"\nCompletado en {elapsed:.1f}s")
        for j, ch in enumerate(chains):
            print(f"  Cadena {j+1}: MH tau accept={ch['mh_final_rate']*100:.1f}%  "
                  f"prop_std={ch['prop_std_final']}")
    return chains


# ── Diagnosticos de convergencia ──────────────────────────────────────────────

def gelman_rubin(chains, key, idx=None):
    '''Estadistico R-hat de Gelman-Rubin (1992).'''
    series = []
    for ch in chains:
        s = ch[key] if idx is None else ch[key][:, idx]
        series.append(np.asarray(s, dtype=float))
    m = len(series); n = len(series[0])
    chain_means = np.array([s.mean() for s in series])
    chain_vars  = np.array([s.var(ddof=1) for s in series])
    grand_mean  = chain_means.mean()
    B   = n / (m-1) * np.sum((chain_means - grand_mean)**2)
    W   = chain_vars.mean()
    var_hat = (n-1)/n * W + B/n
    return float(np.sqrt(var_hat / W)) if W > 1e-15 else float('nan')


def effective_sample_size(series):
    '''ESS = n / (1 + 2*sum(rho_k)).'''
    x   = np.asarray(series, dtype=float)
    n   = len(x); xc = x - x.mean(); var = xc.var()
    if var < 1e-15: return float(n)
    rho_sum = 0.0
    for k in range(1, min(100, n//2)):
        rho_k = np.mean(xc[:n-k] * xc[k:]) / var
        if abs(rho_k) < 0.05: break
        rho_sum += rho_k
    return max(1.0, n / (1.0 + 2.0*rho_sum))


def compute_diagnostics(chains):
    '''
    Tabla R-hat y ESS para todos los parametros escalares y vectoriales.
    Incluye semaforo: OK (< 1.05), ~ (< 1.10), NO (>= 1.10).
    '''
    model = chains[0]['model']
    k     = chains[0]['mu'].shape[1]
    nombres_b = ['beta0(nivel)', 'beta1(pend.)', 'beta2(curv.)', 'beta3(curv2)'][:k]

    checks = [('tau1', None, 'tau1')]
    if model == 'DNSS':
        checks.append(('tau2', None, 'tau2'))
    for i, nb in enumerate(nombres_b):
        checks += [('f_vec', i, f'f({nb})'), ('mu', i, f'mu({nb})'),
                   ('q_vec', i, f'q({nb})')]
    for n in range(min(chains[0]['r_vec'].shape[1], 8)):
        checks.append(('r_vec', n, f'r_vec[{n}]'))

    rows = []
    for key, idx, name in checks:
        rhat     = gelman_rubin(chains, key, idx)
        ess_vals = [effective_sample_size(
            ch[key] if idx is None else ch[key][:, idx]) for ch in chains]
        semaforo = 'OK' if rhat < 1.05 else ('~' if rhat < 1.10 else 'NO')
        rows.append({'Parametro': name, 'Rhat': round(rhat,4),
                     'ESS_min': round(min(ess_vals),0),
                     'ESS_med': round(np.mean(ess_vals),0),
                     'Converge': semaforo})
    return pd.DataFrame(rows)


def print_diagnostics(chains):
    df = compute_diagnostics(chains)
    print(f"\n{'='*58}")
    print(f"  Diagnosticos MCMC — {chains[0]['model']}")
    print(f"{'='*58}")
    print(f"  {'Parametro':<20} {'Rhat':>7} {'ESS_min':>8} {'ESS_med':>8} {'Conv.':>6}")
    print(f"  {'─'*52}")
    for _, row in df.iterrows():
        flag = 'OK' if row['Converge']=='OK' else ('~~' if row['Converge']=='~' else 'XX')
        print(f"  {row['Parametro']:<20} {row['Rhat']:>7.4f} "
              f"{row['ESS_min']:>8.0f} {row['ESS_med']:>8.0f}  {flag}")
    n_bad = (df['Converge'] != 'OK').sum()
    if n_bad == 0:
        print(f"\n  Todos los parametros convergieron (Rhat < 1.05)")
    else:
        print(f"\n  {n_bad} parametros con Rhat >= 1.05 — considerar mas iteraciones")

print("Multi-cadena y diagnosticos definidos")


## CELDA 10: Ejecución del Gibbs sampler

In [ ]:
# ── DNS — 4 cadenas paralelas ─────────────────────────────────────────────
priors_dns = make_priors('DNS', prior_tau_type='uniforme')

chains_dns = run_chains_parallel(
    yields     = yields_data,
    maturities = maturities,
    priors     = priors_dns,
    n_iter     = 3000,    # ajustar segun capacidad: 2000-5000
    n_burnin   = 1000,    # burn-in = ~33%
    n_chains   = 4,
    verbose    = True,
)
print_diagnostics(chains_dns)


In [ ]:
# ── DNSS — 4 cadenas paralelas ────────────────────────────────────────────
priors_dnss = make_priors('DNSS', prior_tau_type='uniforme')

chains_dnss = run_chains_parallel(
    yields     = yields_data,
    maturities = maturities,
    priors     = priors_dnss,
    n_iter     = 3000,
    n_burnin   = 1000,
    n_chains   = 4,
    verbose    = True,
)
print_diagnostics(chains_dnss)


## CELDA 11: Traceplots — Diagnóstico visual de convergencia

Los traceplots muestran la evolución de los parámetros a lo largo de las iteraciones
post burn-in para cada cadena. Una cadena bien mezclada debe:
- **Explorar el espacio:** no quedarse atascada en una región
- **Solaparse entre cadenas:** todas las cadenas deben visitar las mismas regiones
- **Sin tendencia:** la media no debe derivar sistemáticamente


In [ ]:
def plot_traceplots(chains, titulo=''):
    '''Traceplots de parametros clave para todas las cadenas.'''
    model  = chains[0]['model']
    k      = chains[0]['mu'].shape[1]
    n_samp = chains[0]['tau1'].shape[0]
    iter_x = np.arange(1, n_samp+1)

    nombres_b = ['beta0(nivel)', 'beta1(pend.)', 'beta2(curv.)', 'beta3(curv2)'][:k]

    # Seleccion de parametros a graficar
    params_plot = [('tau1', None, 'tau1')]
    if model == 'DNSS':
        params_plot.append(('tau2', None, 'tau2'))
    params_plot += [('f_vec', i, f'f({nombres_b[i]})') for i in range(min(k,2))]
    params_plot += [('q_vec', i, f'sigma_eta({nombres_b[i]})') for i in range(min(k,2))]
    params_plot.append(('r_vec', 0, 'sigma_eps(1Y)'))
    params_plot.append(('r_vec', -1, f'sigma_eps({int(maturities[-1])}Y)'))

    n_params = len(params_plot)
    ncols    = 2
    nrows    = (n_params + 1) // 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.0*nrows), sharex=True)
    axes = axes.flatten()

    for ax, (key, idx, lbl) in zip(axes[:n_params], params_plot):
        for j, ch in enumerate(chains):
            s = ch[key] if idx is None else ch[key][:, idx]
            ax.plot(iter_x, s, color=COLORES_CADENA[j%len(COLORES_CADENA)],
                    alpha=0.75, lw=0.6, label=f'C{j+1}')
        # Linea de media global
        grand = np.concatenate([
            (ch[key] if idx is None else ch[key][:, idx]) for ch in chains])
        ax.axhline(grand.mean(), color='k', lw=1.0, ls='--', alpha=0.5)
        ax.set_title(lbl, fontsize=10)
        ax.set_ylabel('Valor')

    axes[0].legend(fontsize=7, ncol=len(chains), loc='upper right')
    for ax in axes[n_params:]:
        ax.set_visible(False)
    axes[-2].set_xlabel('Iteracion (post burn-in)')
    axes[-1].set_xlabel('Iteracion (post burn-in)')

    fig.suptitle(f'Traceplots — {model} Gibbs Sampler  {titulo}', fontsize=13)
    plt.tight_layout(); plt.show()

    # Resumen de mezcla
    print(f"\nCalidad de mezcla (desviacion std entre cadenas / media global):")
    for key, idx, lbl in params_plot[:6]:
        vals = np.array([(ch[key] if idx is None else ch[key][:, idx]).mean()
                          for ch in chains])
        cv = vals.std() / (abs(vals.mean()) + 1e-10)
        flag = 'OK' if cv < 0.05 else ('~' if cv < 0.15 else 'XX')
        print(f"  {lbl:<22}: CV entre cadenas = {cv:.4f}  {flag}")


plot_traceplots(chains_dns,  '(DNS)')
plot_traceplots(chains_dnss, '(DNSS)')


## CELDA 12: ACF de la cadena — Autocorrelación en las muestras MCMC

La autocorrelación en las muestras del Gibbs sampler reduce el ESS efectivo.
Un ACF que decae rápidamente a cero indica buena mezcla; un decaimiento lento
indica que se necesitan más iteraciones o *thinning* para obtener muestras
aproximadamente independientes.


In [ ]:
def plot_acf_mcmc(chains, n_chain=0, max_lag=60, titulo=''):
    '''ACF de los parametros clave de una cadena.'''
    model  = chains[0]['model']
    k      = chains[0]['mu'].shape[1]
    ch     = chains[n_chain]
    nombres_b = ['beta0', 'beta1', 'beta2', 'beta3'][:k]

    params = [('tau1', None, 'tau1')]
    if model == 'DNSS':
        params.append(('tau2', None, 'tau2'))
    params += [('f_vec', i, f'f({nombres_b[i]})') for i in range(min(k,2))]
    params += [('q_vec', i, f'sigma_eta({nombres_b[i]})') for i in range(min(k,2))]
    params.append(('r_vec', 0, 'sigma_eps(1Y)'))

    n_params = len(params)
    ncols    = 3
    nrows    = (n_params + 2) // 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5*nrows))
    axes = axes.flatten()
    ci   = 1.96 / np.sqrt(len(ch['tau1']))
    lags = np.arange(1, max_lag+1)

    for ax, (key, idx, lbl) in zip(axes[:n_params], params):
        s = ch[key] if idx is None else ch[key][:, idx]
        s_c = s - s.mean(); var = s_c.var()

        if _STATSMODELS:
            _sm_acf(s, lags=max_lag, ax=ax, title=lbl, alpha=0.05, zero=False)
        else:
            # ACF manual
            acf_vals = np.array([np.mean(s_c[:len(s)-lag]*s_c[lag:])/var
                                   for lag in lags])
            ax.bar(lags, acf_vals, color='#1f77b4', alpha=0.7, width=0.8)
            ax.fill_between([0, max_lag+1], -ci, ci, alpha=0.15, color='gray')
            ax.axhline(0, color='k', lw=0.7)
            ax.axhline( ci, color='gray', lw=0.8, ls='--')
            ax.axhline(-ci, color='gray', lw=0.8, ls='--')
            ax.set_title(lbl)

        # ESS de esta cadena
        ess = effective_sample_size(s)
        ax.set_xlabel(f'Lag  (ESS={ess:.0f})', fontsize=8)

    for ax in axes[n_params:]:
        ax.set_visible(False)

    fig.suptitle(f'ACF parametros — Cadena {n_chain+1}  {model}  {titulo}',
                 fontsize=13)
    plt.tight_layout(); plt.show()


plot_acf_mcmc(chains_dns,  n_chain=0, titulo='(DNS)')
plot_acf_mcmc(chains_dnss, n_chain=0, titulo='(DNSS)')


## CELDA 13: Distribuciones posteriores de los parámetros

La ventaja central del enfoque bayesiano: en lugar de estimaciones puntuales,
obtenemos **distribuciones completas** para cada parámetro. Las visualizaciones
incluyen la media posterior, el intervalo de credibilidad al 95%, y (si se especificó)
la forma del prior para comparar qué tanta información aportaron los datos.


In [ ]:
def plot_posteriors(chains, priors, titulo=''):
    '''
    Histogramas de distribuciones posteriores para todos los parametros escalares.
    Muestra: media, IC 95%, prior (si disponible).
    '''
    model  = chains[0]['model']
    k      = chains[0]['mu'].shape[1]
    nombres_b = ['beta0(nivel)', 'beta1(pend.)', 'beta2(curv.)', 'beta3(curv2)'][:k]

    # Concatenar todas las cadenas
    def concat_param(key, idx=None):
        return np.concatenate([
            (ch[key] if idx is None else ch[key][:, idx]) for ch in chains])

    params_list = [
        ('tau1', None, 'tau1', 'azul'),
    ]
    if model == 'DNSS':
        params_list.append(('tau2', None, 'tau2', 'rojo'))
    for i, nb in enumerate(nombres_b):
        params_list.append(('f_vec', i, f'f({nb})', COLORES_FACTOR[i%4]))
        params_list.append(('mu',    i, f'mu({nb})', COLORES_FACTOR[i%4]))
        params_list.append(('q_vec', i, f'sigma_eta({nb})', COLORES_FACTOR[i%4]))

    n_params = len(params_list)
    ncols    = 3
    nrows    = (n_params + 2) // 3
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.2*nrows))
    axes = axes.flatten()

    for ax, (key, idx, lbl, col) in zip(axes[:n_params], params_list):
        samp = concat_param(key, idx)
        q025, q975 = np.percentile(samp, [2.5, 97.5])

        ax.hist(samp, bins=50, density=True, color=col,
                alpha=0.65, edgecolor='white', lw=0.3)

        # Media y IC 95%
        ax.axvline(samp.mean(), color='k', lw=1.5, label=f'Media={samp.mean():.3f}')
        ax.axvline(q025, color='k', lw=0.9, ls='--', alpha=0.7)
        ax.axvline(q975, color='k', lw=0.9, ls='--', alpha=0.7,
                   label=f'IC95%=[{q025:.3f},{q975:.3f}]')

        # KDE suavizada
        try:
            x_kde = np.linspace(samp.min()*0.95, samp.max()*1.05, 200)
            ax.plot(x_kde, stats.gaussian_kde(samp)(x_kde), color=col, lw=2.0)
        except Exception:
            pass

        ax.set_title(lbl, fontsize=10)
        ax.set_xlabel('Valor'); ax.set_ylabel('Densidad')
        ax.legend(fontsize=7)

    for ax in axes[n_params:]:
        ax.set_visible(False)

    fig.suptitle(f'Distribuciones posteriores — {model}  {titulo}', fontsize=13)
    plt.tight_layout(); plt.show()

    # Tabla resumen posterior
    print(f"\n── Resumen posterior — {model} ───────────────────────────────────────")
    print(f"  {'Parametro':<22} {'Media':>8} {'Std':>7} {'Q2.5':>8} {'Q50':>8} {'Q97.5':>8}")
    print(f"  {'─'*62}")
    for key, idx, lbl, _ in params_list:
        samp = concat_param(key, idx)
        q025, q50, q975 = np.percentile(samp, [2.5, 50, 97.5])
        print(f"  {lbl:<22} {samp.mean():>8.4f} {samp.std():>7.4f} "
              f"{q025:>8.4f} {q50:>8.4f} {q975:>8.4f}")


plot_posteriors(chains_dns,  priors_dns,  '(DNS)')
plot_posteriors(chains_dnss, priors_dnss, '(DNSS)')


## CELDA 14: Posterior de $\tau$ y ubicación del hump

A diferencia del enfoque KF donde $\tau$ es un valor único (seleccionado por Grid Search),
aquí tenemos la **distribución completa** de $\tau$ con su incertidumbre. El hump de
la curvatura ocurre en $m^* = 1.793\tau_1$ años.


In [ ]:
def plot_posterior_tau(chains, maturities, titulo=''):
    '''Distribucion posterior de tau y del hump location.'''
    model   = chains[0]['model']
    tau1    = np.concatenate([ch['tau1'] for ch in chains])
    hump1   = 1.793 * tau1
    mat_max = maturities.max()

    n_cols = 2 if model == 'DNS' else 4
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 4.5))
    if n_cols == 2: axes = list(axes)

    def _dist_panel(ax, samp, lbl, col, xlab='Anos'):
        q025, q975 = np.percentile(samp, [2.5, 97.5])
        x_kde = np.linspace(samp.min()*0.9, samp.max()*1.1, 200)
        try: kde = stats.gaussian_kde(samp)(x_kde)
        except: kde = np.ones_like(x_kde)
        ax.fill_between(x_kde, kde, alpha=0.35, color=col)
        ax.plot(x_kde, kde, color=col, lw=2.0)
        ax.axvline(samp.mean(), color='k', lw=1.5, ls='-',
                   label=f'Media={samp.mean():.3f}')
        ax.axvline(q025, color='k', lw=0.9, ls='--')
        ax.axvline(q975, color='k', lw=0.9, ls='--',
                   label=f'IC95%=[{q025:.3f},{q975:.3f}]')
        ax.set_title(lbl); ax.set_xlabel(xlab); ax.set_ylabel('Densidad')
        ax.legend(fontsize=8)

    _dist_panel(axes[0], tau1,  'Posterior tau1', COLORES_TAU[0])
    _dist_panel(axes[1], hump1, 'Hump beta2 (anos)', COLORES_TAU[0], 'Anos')

    if model == 'DNSS':
        tau2  = np.concatenate([ch['tau2'] for ch in chains])
        hump2 = 1.793 * tau2
        _dist_panel(axes[2], tau2,  'Posterior tau2', COLORES_TAU[1])
        _dist_panel(axes[3], hump2, 'Hump beta3 (anos)', COLORES_TAU[1], 'Anos')

    fig.suptitle(f'Posterior de tau y hump location — {model}  {titulo}', fontsize=13)
    plt.tight_layout(); plt.show()


plot_posterior_tau(chains_dns,  maturities, '(DNS)')
plot_posterior_tau(chains_dnss, maturities, '(DNSS)')


## CELDA 15: Factores latentes y curva ajustada

En el contexto bayesiano, los "estados filtrados" son reemplazados por la
**media posterior de $\beta_{1:T}$** (promedio sobre todas las trayectorias muestreadas)
con bandas de credibilidad al 95% basadas en los percentiles de las muestras MCMC.
Estas bandas son más rigurosas que los intervalos del KF porque integran la incertidumbre
tanto del estado como de los parámetros $\theta$ y $\tau$.


In [ ]:
def extract_posterior_betas(chains):
    '''
    Extrae estadisticos de la distribucion posterior de beta_t.
    Combina todas las cadenas (cada iteracion es una trayectoria distinta).

    Retorna
    -------
    beta_mean  : (T, k)    media posterior
    beta_q025  : (T, k)    percentil 2.5%
    beta_q975  : (T, k)    percentil 97.5%
    beta_std   : (T, k)    desviacion estandar posterior
    '''
    # Forma: (n_chains * n_store, T, k)
    all_betas = np.concatenate([ch['betas'] for ch in chains], axis=0)
    return (all_betas.mean(axis=0),
            np.percentile(all_betas, 2.5, axis=0),
            np.percentile(all_betas, 97.5, axis=0),
            all_betas.std(axis=0))


def plot_factores_bayes(chains, dates, titulo=''):
    '''Factores latentes: media posterior + IC 95% bayesiano.'''
    model  = chains[0]['model']
    k      = chains[0]['mu'].shape[1]
    beta_mean, beta_q025, beta_q975, beta_std = extract_posterior_betas(chains)

    nombres = (['beta0 Nivel', 'beta1 Pendiente', 'beta2 Curvatura']
               + (['beta3 2a Curvatura'] if k==4 else []))

    fig, axes = plt.subplots(k, 1, figsize=(14, 3.0*k), sharex=True)
    if k == 1: axes = [axes]

    for i, (ax, nom, col) in enumerate(zip(axes, nombres, COLORES_FACTOR)):
        ax.fill_between(dates, beta_q025[:,i], beta_q975[:,i],
                        alpha=0.20, color=col, label='IC 95% bayesiano')
        ax.plot(dates, beta_mean[:,i], color=col, lw=1.8, label='Media posterior')
        ax.axhline(0, color='k', lw=0.6, ls=':')
        ax.set_ylabel(nom, fontsize=10)
        ax.legend(loc='upper right', ncol=2)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    axes[-1].set_xlabel('Fecha')
    fig.suptitle(f'Factores latentes — {model} Bayesiano  {titulo}', fontsize=13)
    plt.tight_layout(); plt.show()

    # Estadisticos descriptivos
    print(f"\n── Estadisticos de beta_t (media posterior) — {model} ───────────")
    print(f"  {'Factor':<16} {'Media':>8} {'Std':>7} {'Q2.5':>8} {'Q97.5':>8} {'rho(1)':>8}")
    print(f"  {'─'*56}")
    for i, nom in enumerate(nombres):
        s  = beta_mean[:,i]
        ac = np.corrcoef(s[:-1], s[1:])[0,1]
        q025 = beta_q025[:,i].mean(); q975 = beta_q975[:,i].mean()
        print(f"  {nom[:15]:<16} {s.mean():>8.3f} {s.std():>7.3f} "
              f"{q025:>8.3f} {q975:>8.3f} {ac:>8.4f}")


def compute_posterior_fit(chains, yields, maturities):
    '''
    Calcula rendimientos ajustados y residuos usando la media posterior.
    Usa la media posterior de tau y beta.
    '''
    model = chains[0]['model']
    beta_mean, _, _, _ = extract_posterior_betas(chains)

    tau1_all = np.concatenate([ch['tau1'] for ch in chains])
    tau1_pm  = tau1_all.mean()
    tau2_pm  = None
    if model == 'DNSS':
        tau2_all = np.concatenate([ch['tau2'] for ch in chains])
        tau2_pm  = tau2_all.mean()

    H_pm   = get_H(maturities, tau1_pm, tau2_pm, model)
    fitted = (H_pm @ beta_mean.T).T
    resid  = yields - fitted

    rmse_total = np.sqrt(np.mean(resid**2))
    mae_total  = np.mean(np.abs(resid))
    rmse_mat   = np.sqrt(np.mean(resid**2, axis=0))
    mae_mat    = np.mean(np.abs(resid),    axis=0)

    return {
        'model'     : model,
        'beta_mean' : beta_mean,
        'tau1_pm'   : tau1_pm, 'tau2_pm': tau2_pm,
        'H_pm'      : H_pm,
        'fitted'    : fitted, 'residuals': resid,
        'rmse_total': rmse_total, 'mae_total': mae_total,
        'rmse_mat'  : rmse_mat,   'mae_mat'  : mae_mat,
        'dates'     : dates_dt,   'yields'   : yields,
        'maturities': maturities,
        # claves que esperan las funciones del KF notebook (compatibilidad)
        'beta_smoothed': beta_mean,
        'P_smoothed'   : np.zeros((len(dates_dt), chains[0]['mu'].shape[1],
                                    chains[0]['mu'].shape[1])),
        'beta_filtered': beta_mean,
        'innovations'  : resid,
        'tau1_t'       : np.full(len(dates_dt), tau1_pm),
        'tau2_t'       : np.full(len(dates_dt), tau2_pm) if tau2_pm else np.full(len(dates_dt), np.nan),
        'F_type'       : 'diagonal',
        'window_size'  : None,
        'mu'           : np.concatenate([ch['mu'] for ch in chains]).mean(axis=0),
        'F'            : np.diag(np.concatenate([ch['f_vec'] for ch in chains]).mean(axis=0)),
        'Q'            : np.diag(np.concatenate([ch['q_vec'] for ch in chains]).mean(axis=0)),
        'R_diag'       : np.concatenate([ch['r_vec'] for ch in chains]).mean(axis=0),
        'sse_grid'     : None,
        'H_arr'        : np.tile(H_pm, (len(dates_dt), 1, 1)),
    }


plot_factores_bayes(chains_dns,  dates_dt, '(DNS)')
plot_factores_bayes(chains_dnss, dates_dt, '(DNSS)')


## CELDA 16: Análisis post-ajuste — Residuos y ajuste in-sample

Reutilizamos el mismo análisis robusto del notebook de Kalman Filter.
Los residuos aquí son $\hat{\varepsilon}_{t,n} = y_{t,n} - H(\bar{\tau})\bar{\beta}_{t|Y}$
donde $\bar{\tau}$ y $\bar{\beta}_{t|Y}$ son las medias posteriores.


In [ ]:
# Calcular ajuste posterior para DNS y DNSS
fit_dns_b  = compute_posterior_fit(chains_dns,  yields_data, maturities)
fit_dnss_b = compute_posterior_fit(chains_dnss, yields_data, maturities)

# ── Tabla comparativa ─────────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════════╗")
print("║    Comparacion DNS vs DNSS — Enfoque Bayesiano      ║")
print("╠═══════════════════╦══════════════╦═══════════════════╣")
print("║ Metrica           ║    DNS       ║     DNSS          ║")
print("╠═══════════════════╬══════════════╬═══════════════════╣")
metricas = [('RMSE total (pb)','rmse_total',100,'.4f'),
            ('MAE total (pb)', 'mae_total', 100,'.4f'),
            ('tau1 posterior', 'tau1_pm',     1,'.3f'),
            ('tau2 posterior', 'tau2_pm',      1,'.3f')]
for label, attr, scale, fmt in metricas:
    vd  = (fit_dns_b[attr]  or 0) * scale
    vss = (fit_dnss_b[attr] or 0) * scale
    print(f"║ {label:<17} ║ {vd:{fmt}:>12} ║ {vss:{fmt}:>17} ║")
print("╚═══════════════════╩══════════════╩═══════════════════╝")

print(f"\n── RMSE por plazo (pb) ─────────────────────────────────────")
print(f"  {'Plazo':>5}  {'DNS':>8}  {'DNSS':>8}  {'Dif':>8}  Mejor")
for i, m in enumerate(maturities):
    rd  = fit_dns_b['rmse_mat'][i]  * 100
    rss = fit_dnss_b['rmse_mat'][i] * 100
    print(f"  {m:>4.0f}Y  {rd:>8.4f}  {rss:>8.4f}  {rd-rss:>+8.4f}  "
          f"{'DNS' if rd<rss else 'DNSS'}")


In [ ]:
# ── Analisis de residuos — mismas funciones que el KF notebook ───────────────
# (copiar aqui las funciones analisis_residuos, plot_heatmap_residuos,
#  plot_curvas_ajustadas del notebook DNS_DNSS_KalmanFilter_Adaptativo.ipynb
#  o importarlas si estan en el mismo entorno)

# Implementacion compacta compatible con el dict de resultados bayesianos:

from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap

def plot_heatmap_residuos_b(fit, titulo=''):
    resid = fit['residuals'] * 100
    dates = fit['dates']
    mats  = fit['maturities']
    T     = len(dates)
    fig, ax = plt.subplots(figsize=(14, 4.5))
    vmax = np.percentile(np.abs(resid), 97)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    cmap = LinearSegmentedColormap.from_list('bwr_soft',['#2166ac','#f7f7f7','#d6604d'])
    im = ax.imshow(resid.T, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest')
    n_t = min(10, T//20+1)
    t_ticks = np.linspace(0, T-1, n_t, dtype=int)
    ax.set_xticks(t_ticks)
    ax.set_xticklabels([str(dates[i])[:7] for i in t_ticks], rotation=45, ha='right')
    ax.set_yticks(range(len(mats)))
    ax.set_yticklabels([f'{m:.0f}Y' for m in mats])
    ax.set_title(f'Residuos (pb) — {fit["model"]} Bayesiano  {titulo}  '
                 f'[rojo=subestima, azul=sobreestima]')
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label='Residuo (pb)')
    plt.tight_layout(); plt.show()


def analisis_residuos_b(fit, titulo=''):
    model = fit['model']; resid = fit['residuals']
    mats  = fit['maturities']; T, N = resid.shape; pb = resid*100

    rmse_m = np.sqrt(np.mean(resid**2, axis=0))*100
    mae_m  = np.mean(np.abs(resid),    axis=0)*100
    bias_m = resid.mean(axis=0)*100

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    x = np.arange(N); w = 0.35
    axes[0].bar(x-w/2, rmse_m, w, label='RMSE', color='#1f77b4', alpha=0.82)
    axes[0].bar(x+w/2, mae_m,  w, label='MAE',  color='#d62728', alpha=0.82)
    axes[0].set_xticks(x); axes[0].set_xticklabels([f'{m:.0f}Y' for m in mats])
    axes[0].set_xlabel('Plazo'); axes[0].set_ylabel('pb')
    axes[0].set_title(f'RMSE y MAE por plazo — {model} Bayes  {titulo}')
    axes[0].legend()

    axes[1].bar(x, bias_m, color=['#2ca02c' if b>=0 else '#d62728' for b in bias_m], alpha=0.8)
    axes[1].axhline(0, color='k', lw=0.8)
    axes[1].set_xticks(x); axes[1].set_xticklabels([f'{m:.0f}Y' for m in mats])
    axes[1].set_xlabel('Plazo'); axes[1].set_ylabel('pb'); axes[1].set_title('Sesgo medio')

    corr_r = np.corrcoef(resid.T)
    im = axes[2].imshow(corr_r, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[2].set_xticks(range(N)); axes[2].set_yticks(range(N))
    axes[2].set_xticklabels([f'{m:.0f}Y' for m in mats], rotation=45, fontsize=8)
    axes[2].set_yticklabels([f'{m:.0f}Y' for m in mats], fontsize=8)
    axes[2].set_title('Correlacion entre residuos')
    plt.colorbar(im, ax=axes[2], fraction=0.04)
    for ii in range(N):
        for jj in range(N):
            axes[2].text(jj, ii, f'{corr_r[ii,jj]:.2f}', ha='center',
                         va='center', fontsize=6)
    plt.tight_layout(); plt.show()

    print(f"\n── Residuos — {model} Bayesiano  {titulo} ──────────────────────────────")
    print(f"  {'Plazo':>5}  {'RMSE':>8}  {'Sesgo':>8}  {'JB p':>8}")
    print(f"  {'─'*40}")
    for i, m in enumerate(mats):
        r = pb[:,i]; jb = stats.jarque_bera(r)
        print(f"  {m:>4.0f}Y  {np.sqrt(np.mean(r**2)):>8.3f}  {r.mean():>+8.3f}  {jb.pvalue:>8.4f}")


plot_heatmap_residuos_b(fit_dns_b,  '(DNS)')
analisis_residuos_b(fit_dns_b,  '(DNS)')
plot_heatmap_residuos_b(fit_dnss_b, '(DNSS)')
analisis_residuos_b(fit_dnss_b, '(DNSS)')


## CELDA 17: Comparativa Bayesiano vs Kalman Filter

Una de las ventajas del enfoque bayesiano es la cuantificación completa de la
incertidumbre. Las bandas de credibilidad de los factores son más amplias que
los intervalos del KF porque incorporan la incertidumbre de los parámetros θ y τ.


In [ ]:
def plot_comparativa_bayes_kf(chains, fit_kf, dates, titulo=''):
    '''
    Compara bandas de credibilidad bayesianas vs intervalos KF para los factores.
    fit_kf debe ser el resultado de fit_model() del notebook de KF.
    '''
    model  = chains[0]['model']
    k      = chains[0]['mu'].shape[1]
    beta_mean, beta_q025, beta_q975, _ = extract_posterior_betas(chains)
    nombres = ['beta0 Nivel','beta1 Pendiente','beta2 Curvatura','beta3 2a Curvatura'][:k]

    fig, axes = plt.subplots(k, 1, figsize=(14, 3.0*k), sharex=True)
    if k == 1: axes = [axes]

    for i, (ax, nom, col) in enumerate(zip(axes, nombres, COLORES_FACTOR)):
        # Banda bayesiana
        ax.fill_between(dates, beta_q025[:,i], beta_q975[:,i],
                        alpha=0.25, color=col, label='IC 95% Bayes')
        ax.plot(dates, beta_mean[:,i], color=col, lw=1.8, label='Media posterior (Bayes)')

        # KF smoothed si disponible
        if fit_kf is not None and 'beta_smoothed' in fit_kf:
            bs_kf = fit_kf['beta_smoothed']
            Ps_kf = fit_kf.get('P_smoothed', None)
            ax.plot(dates, bs_kf[:,i], color=col, lw=0.9, ls='--',
                    alpha=0.70, label='RTS smoother (KF)')
            if Ps_kf is not None:
                sd_kf = np.sqrt(np.maximum(Ps_kf[:,i,i], 0))
                ax.fill_between(dates, bs_kf[:,i]-1.96*sd_kf, bs_kf[:,i]+1.96*sd_kf,
                                alpha=0.10, color='gray', label='IC 95% KF')

        ax.axhline(0, color='k', lw=0.6, ls=':')
        ax.set_ylabel(nom); ax.legend(loc='upper right', ncol=4, fontsize=8)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    axes[-1].set_xlabel('Fecha')
    fig.suptitle(f'Comparativa Bayes vs KF — {model}  {titulo}', fontsize=13)
    plt.tight_layout(); plt.show()


# Si tienes resultados KF disponibles (res_dns del otro notebook):
# plot_comparativa_bayes_kf(chains_dns, res_dns, dates_dt, '(DNS)')
# De lo contrario, graficar solo el bayesiano:
plot_factores_bayes(chains_dns,  dates_dt, '(DNS)')
plot_factores_bayes(chains_dnss, dates_dt, '(DNSS)')


## CELDA 18: Función principal `fit_dns_bayes()`

Interfaz de alto nivel que orquesta el pipeline completo bayesiano.
Compatible con el mismo estilo de uso que `fit_dns_kalman()` del otro notebook.


In [ ]:
def fit_dns_bayes(
    dates,
    yields,
    maturities,
    # ── Modelo ───────────────────────────────────────────────────────────────
    model            = 'DNS',
    # ── MCMC ─────────────────────────────────────────────────────────────────
    n_iter           = 3000,
    n_burnin         = 1000,
    n_chains         = 4,
    # ── Modo de tau ───────────────────────────────────────────────────────────
    tau_mode         = 'estatico',   # 'estatico' | 'ventana' | 'ar1'
    tau_window_size  = None,         # para tau_mode='ventana'
    tau_ar1_prop_std = 0.05,         # para tau_mode='ar1'
    # ── Priors de tau ────────────────────────────────────────────────────────
    prior_tau_type   = 'uniforme',
    tau1_mu          = 1.78,  tau1_sd = 0.40,
    tau2_mu          = 0.60,  tau2_sd = 0.08,
    tau_lo           = 0.15,  tau_hi  = 5.0,
    # ── MH ───────────────────────────────────────────────────────────────────
    prop_std_init    = None,
    # ── Mejoras R ────────────────────────────────────────────────────────────
    obs_weights      = None,
    r_mode           = 'libre',
    group_thresholds = (3.0, 10.0),
    # ── Control ──────────────────────────────────────────────────────────────
    plot             = True,
    verbose          = True,
    n_jobs           = -1,
):
    '''
    Ajuste bayesiano DNS o DNSS via Gibbs sampler con FFBS.
    Soporta tres modos de modelado de tau (v2).

    Parametros de tau_mode
    ----------------------
    tau_mode : str
        'estatico'  — tau global, unico para toda la muestra. Posterior via MH
                      (Roberts & Rosenthal 2001). Comportamiento original v1.
                      Pros: el mas rapido, IC de tau bien definido.
                      Contras: asume forma de la curva constante en el tiempo.

        'ventana'   — tau cambia entre ventanas rodantes de tamano tau_window_size.
                      En cada ventana se corre un mini-sampler que actualiza tau.
                      Pros: tau semi-adaptativo, sin cambio de modelo.
                      Contras: intervalos de tau mas amplios (menos datos por ventana).

        'ar1'       — tau_t sigue un proceso AR(1) latente en log-escala:
                        x_t = log(tau_t),
                        x_t = mu_x + phi_x*(x_{t-1}-mu_x) + sigma_x*eps_t
                      Se muestrea x_1:T via MH por periodo con prior AR(1).
                      Hiperparametros phi_x y sigma_x tienen priors conjugadas.
                      Pros: tau totalmente dinamico, captura cambios de regimen.
                      Contras: el mas lento; requiere mas iteraciones para convergencia.

    tau_window_size : int (solo para tau_mode='ventana')
        Tamano de cada ventana en periodos. None -> max(30, T//4).

    tau_ar1_prop_std : float (solo para tau_mode='ar1')
        Std de la propuesta MH por periodo para log(tau_t). Default 0.05.
        Valores menores -> tasa de aceptacion mas alta pero mezcla mas lenta.

    Resto de parametros
    -------------------
    (igual que v1 — ver docstring de make_priors y run_chain)

    Retorna
    -------
    dict con:
      "chains"       : lista de n_chains dicts con muestras MCMC
      "fit"          : dict de resultados post-ajuste
      "diagnostics"  : DataFrame con R-hat y ESS
      "priors"       : priors usados
      "model"        : "DNS" o "DNSS"
      "tau_mode"     : modo de tau usado

    Claves adicionales en "fit" para tau_mode != 'estatico':
      "tau1_t"   : (T,)  secuencia temporal de tau1 (media posterior)
      "tau2_t"   : (T,)  secuencia temporal de tau2 (solo DNSS)

    Ejemplos
    --------
    # Modo 1 — tau estatico (clasico)
    res = fit_dns_bayes(dates, yields, mats, model='DNS',
                        tau_mode='estatico')

    # Modo 2 — tau semi-adaptativo por ventanas
    res = fit_dns_bayes(dates, yields, mats, model='DNSS',
                        tau_mode='ventana', tau_window_size=52)

    # Modo 3 — tau AR(1) latente
    res = fit_dns_bayes(dates, yields, mats, model='DNS',
                        tau_mode='ar1', tau_ar1_prop_std=0.04,
                        n_iter=5000, n_burnin=2000)

    # Comparar los tres modos
    for modo in ['estatico', 'ventana', 'ar1']:
        r = fit_dns_bayes(dates, yields, mats, tau_mode=modo, plot=False)
        print(f"{modo}: RMSE={r['fit']['rmse_total']*100:.3f}pb")
    '''
    # ── Validaciones ──────────────────────────────────────────────────────────
    if isinstance(yields, pd.DataFrame):
        yields = yields.values
    yields     = np.asarray(yields, dtype=float)
    maturities = np.asarray(maturities, dtype=float)
    if not isinstance(dates, pd.DatetimeIndex):
        dates = pd.DatetimeIndex(dates)

    assert yields.ndim == 2,   "yields debe ser (T, N)"
    T, N = yields.shape
    assert len(maturities) == N
    assert model   in ('DNS', 'DNSS')
    assert tau_mode in ('estatico', 'ventana', 'ar1'), \
        "tau_mode debe ser 'estatico', 'ventana' o 'ar1'"

    # ── Priors ────────────────────────────────────────────────────────────────
    tau1_init, tau2_init = ols_init_tau(yields, maturities, model)
    if tau_mode == 'ar1':
        priors = make_priors_ar1(model, tau1_init, tau2_init,
                                 prior_tau_type=prior_tau_type,
                                 tau1_mu=tau1_mu, tau1_sd=tau1_sd,
                                 tau2_mu=tau2_mu, tau2_sd=tau2_sd,
                                 tau_lo=tau_lo, tau_hi=tau_hi)
    else:
        priors = make_priors(model, prior_tau_type,
                             tau1_mu, tau1_sd, tau2_mu, tau2_sd, tau_lo, tau_hi)

    # ── Sampler ───────────────────────────────────────────────────────────────
    chains = run_chains_parallel(
        yields, maturities, priors, n_iter, n_burnin,
        n_chains=n_chains, prop_std_init=prop_std_init,
        obs_weights=obs_weights, r_mode=r_mode,
        group_thresholds=group_thresholds,
        tau_mode=tau_mode,
        tau_window_size=tau_window_size,
        tau_ar1_prop_std=tau_ar1_prop_std,
        n_jobs=n_jobs, verbose=verbose,
    )

    # ── Diagnosticos ──────────────────────────────────────────────────────────
    diag = compute_diagnostics(chains)
    if verbose:
        print_diagnostics(chains)

    # ── Ajuste posterior ──────────────────────────────────────────────────────
    beta_mean, beta_q025, beta_q975, beta_std = extract_posterior_betas(chains)

    tau1_all = np.concatenate([ch['tau1'] for ch in chains])
    tau1_pm  = tau1_all.mean()
    tau2_pm  = None
    if model == 'DNSS':
        tau2_all = np.concatenate([ch['tau2'] for ch in chains])
        tau2_pm  = tau2_all.mean()

    # Secuencia temporal de tau (para modos ventana y ar1)
    tau1_t = np.full(T, tau1_pm)
    tau2_t = np.full(T, tau2_pm) if tau2_pm else np.full(T, np.nan)

    if tau_mode == 'ar1':
        seq1_all = np.concatenate([ch['tau1_seq'] for ch in chains], axis=0)
        tau1_t   = seq1_all.mean(axis=0)
        tau1_pm  = tau1_t.mean()
        if model == 'DNSS':
            seq2_all = np.concatenate([ch['tau2_seq'] for ch in chains], axis=0)
            tau2_t   = seq2_all.mean(axis=0)
            tau2_pm  = tau2_t.mean()

    # H usando tau medio (para metricas de ajuste)
    H_pm   = get_H(maturities, tau1_pm, tau2_pm, model)
    fitted = (H_pm @ beta_mean.T).T
    resid  = yields - fitted

    fit = {
        'model'      : model,
        'tau_mode'   : tau_mode,
        'beta_mean'  : beta_mean,
        'beta_q025'  : beta_q025,
        'beta_q975'  : beta_q975,
        'beta_std'   : beta_std,
        'tau1_pm'    : tau1_pm,
        'tau2_pm'    : tau2_pm,
        'tau1_t'     : tau1_t,
        'tau2_t'     : tau2_t,
        'fitted'     : fitted,
        'residuals'  : resid,
        'rmse_total' : float(np.sqrt(np.mean(resid**2))),
        'mae_total'  : float(np.mean(np.abs(resid))),
        'rmse_mat'   : np.sqrt(np.mean(resid**2, axis=0)),
        'mae_mat'    : np.mean(np.abs(resid), axis=0),
        'dates'      : dates,
        'yields'     : yields,
        'maturities' : maturities,
        # compatibilidad con funciones del KF notebook
        'beta_smoothed': beta_mean,
        'beta_filtered': beta_mean,
        'P_smoothed'   : np.zeros((T, priors['k'], priors['k'])),
        'H_arr'        : np.tile(H_pm, (T, 1, 1)),
        'F_type'       : 'diagonal',
        'mu'           : np.concatenate([ch['mu']    for ch in chains]).mean(axis=0),
        'F'            : np.diag(np.concatenate([ch['f_vec'] for ch in chains]).mean(axis=0)),
        'Q'            : np.diag(np.concatenate([ch['q_vec'] for ch in chains]).mean(axis=0)),
        'R_diag'       : np.concatenate([ch['r_vec'] for ch in chains]).mean(axis=0),
        'innovations'  : resid,
        'sse_grid'     : None,
        'window_size'  : tau_window_size,
    }

    if verbose:
        print(f"\nRMSE total : {fit['rmse_total']*100:.4f} pb")
        print(f"tau_mode   : {tau_mode}")
        print(f"tau1 media : {tau1_pm:.3f}  (hump~{1.793*tau1_pm:.1f}Y)")
        if tau2_pm:
            print(f"tau2 media : {tau2_pm:.3f}  (hump~{1.793*tau2_pm:.1f}Y)")

    # ── Graficos ──────────────────────────────────────────────────────────────
    if plot:
        plot_traceplots(chains)
        plot_acf_mcmc(chains)
        plot_posteriors(chains, priors)
        plot_posterior_tau(chains, maturities)
        plot_factores_bayes(chains, dates)
        if hasattr(fit, '__getitem__'):
            plot_heatmap_residuos_b(fit)
            analisis_residuos_b(fit)

    return {'chains': chains, 'fit': fit, 'diagnostics': diag,
            'priors': priors, 'model': model, 'tau_mode': tau_mode}


print("fit_dns_bayes() v2 definido — tau_mode: 'estatico' | 'ventana' | 'ar1'")
print()
print("Modos disponibles:")
print("  fit_dns_bayes(..., tau_mode='estatico')          # un tau global (default)")
print("  fit_dns_bayes(..., tau_mode='ventana', tau_window_size=52)")
print("  fit_dns_bayes(..., tau_mode='ar1', tau_ar1_prop_std=0.04)")


## CELDA 19: Experimento — Opciones A y C en el enfoque bayesiano

Las mismas mejoras para plazos largos que en el KF, ahora en el contexto bayesiano:

- **Opción A** (`r_mode='grupos'`): solo 3 hiperparámetros de varianza de medición
  en lugar de N, con posterior IG conjugada por grupo de plazos.
- **Opción C** (`obs_weights`): reduce el `prior_b` efectivo de la posterior IG para
  los plazos ponderados, haciendo que el sampler prefiera varianzas menores en esos plazos.


In [ ]:
# Pesos para plazos largos (Opcion C)
obs_weights_b = np.where(maturities >= 10, 2.0, 1.0)
print(f"Pesos por plazo: {dict(zip([f'{m:.0f}Y' for m in maturities], obs_weights_b))}")

print("\n" + "="*60)
print("  DNSS Bayesiano — R grupos + pesos (Opciones A + C)")
res_dnss_AC = fit_dns_bayes(
    dates            = dates_dt,
    yields           = yields_data,
    maturities       = maturities,
    model            = 'DNSS',
    n_iter           = 3000,
    n_burnin         = 1000,
    n_chains         = 4,
    obs_weights      = obs_weights_b,     # Opcion C
    r_mode           = 'grupos',          # Opcion A
    group_thresholds = (3.0, 10.0),
    plot             = False,
    verbose          = True,
)

# Comparativa
print(f"\n{'='*60}")
print(f"  Comparativa RMSE — DNSS base vs DNSS Opciones A+C")
print(f"{'='*60}")
print(f"  {'Plazo':>5}  {'DNSS-base':>10}  {'DNSS-AC':>9}  {'Delta':>8}")
for i, m in enumerate(maturities):
    rb  = fit_dnss_b['rmse_mat'][i]            * 100
    rac = res_dnss_AC['fit']['rmse_mat'][i]    * 100
    print(f"  {m:>4.0f}Y  {rb:>10.3f}  {rac:>9.3f}  {rac-rb:>+8.3f}")


## Visualización: τ_t temporal (modos ventana y AR(1))

Función para comparar la trayectoria de τ entre los tres modos.


In [ ]:
def plot_tau_temporal(results_dict, dates, maturities, titulo=''):
    '''
    Compara la evolucion temporal de tau entre distintos modos.

    Parametros
    ----------
    results_dict : dict  {label: res}  donde res es el retorno de fit_dns_bayes()
    '''
    fig, axes = plt.subplots(len(results_dict), 1,
                              figsize=(14, 3.5*len(results_dict)), sharex=True)
    if len(results_dict) == 1:
        axes = [axes]

    for ax, (label, res) in zip(axes, results_dict.items()):
        fit  = res['fit']
        mode = fit.get('tau_mode', 'estatico')
        tau1_t = fit['tau1_t']
        tau2_t = fit.get('tau2_t', None)

        if mode == 'ar1':
            # Mostrar trayectoria completa con IC
            chains = res['chains']
            seq1 = np.concatenate([ch['tau1_seq'] for ch in chains], axis=0)
            q025 = np.percentile(seq1, 2.5,  axis=0)
            q975 = np.percentile(seq1, 97.5, axis=0)
            ax.fill_between(dates, q025, q975,
                            alpha=0.20, color=COLORES_TAU[0], label='IC 95%')
            ax.plot(dates, tau1_t, color=COLORES_TAU[0], lw=1.5, label='Media posterior tau1_t')
        elif mode == 'ventana':
            ax.step(dates, tau1_t, color=COLORES_TAU[0], lw=1.5,
                    where='post', label='tau1 por ventana')
        else:
            chains = res['chains']
            tau1_all = np.concatenate([ch['tau1'] for ch in chains])
            q025 = np.percentile(tau1_all, 2.5)
            q975 = np.percentile(tau1_all, 97.5)
            ax.axhline(tau1_t[0], color=COLORES_TAU[0], lw=1.8,
                       label=f'tau1 global = {tau1_t[0]:.3f}')
            ax.fill_between(dates, q025, q975,
                            alpha=0.15, color=COLORES_TAU[0],
                            label=f'IC95%=[{q025:.2f},{q975:.2f}]')

        if tau2_t is not None and not np.all(np.isnan(tau2_t)):
            ax2 = ax.twinx()
            ax2.plot(dates, tau2_t, color=COLORES_TAU[1], lw=1.2,
                     ls='--', alpha=0.7, label='tau2')
            ax2.set_ylabel('tau2', color=COLORES_TAU[1], fontsize=9)

        hump_t = 1.793 * tau1_t
        ax.set_title(f'{label}  [tau_mode=\'{mode}\']  '
                     f'hump medio={np.mean(hump_t):.2f}Y')
        ax.set_ylabel('tau1')
        ax.legend(fontsize=8); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    axes[-1].set_xlabel('Fecha')
    fig.suptitle(f'Evolucion temporal de tau — {titulo}', fontsize=13)
    plt.tight_layout(); plt.show()


def comparar_modos_tau(yields, maturities, dates, model='DNS',
                       n_iter=2000, n_burnin=600, n_chains=2,
                       window_size=52, ar1_prop_std=0.04,
                       verbose=False):
    '''
    Corre los tres modos de tau y compara RMSE, tau medio y graficos.

    Util para decidir que modo usar en produccion.
    '''
    modos = {
        'Estatico'  : dict(tau_mode='estatico'),
        'Ventana'   : dict(tau_mode='ventana', tau_window_size=window_size),
        'AR(1)'     : dict(tau_mode='ar1',     tau_ar1_prop_std=ar1_prop_std),
    }
    resultados = {}
    for nombre, kw in modos.items():
        print(f"\n{'='*50}")
        print(f"  Modo: {nombre}")
        r = fit_dns_bayes(dates, yields, maturities,
                          model=model, n_iter=n_iter, n_burnin=n_burnin,
                          n_chains=n_chains, plot=False, verbose=verbose, **kw)
        resultados[nombre] = r
        print(f"  RMSE = {r['fit']['rmse_total']*100:.4f} pb  "
              f"tau1 media = {r['fit']['tau1_pm']:.3f}")

    # Tabla comparativa
    print(f"\n{'='*60}")
    print(f"  Comparativa de modos de tau — {model}")
    print(f"{'='*60}")
    print(f"  {'Modo':<12}  {'RMSE(pb)':>9}  {'tau1 med':>9}  "
          f"{'hump(Y)':>8}  {'Rhat_max':>9}")
    print(f"  {'─'*55}")
    for nombre, r in resultados.items():
        rmse = r['fit']['rmse_total'] * 100
        t1   = r['fit']['tau1_pm']
        rhat = r['diagnostics']['Rhat'].max()
        print(f"  {nombre:<12}  {rmse:>9.4f}  {t1:>9.3f}  "
              f"{1.793*t1:>8.2f}  {rhat:>9.4f}")

    # Grafico comparativo
    plot_tau_temporal(resultados, dates, maturities, titulo=model)
    return resultados


print("Funciones de comparacion de modos tau definidas")
print()
print("Uso: comparar_modos_tau(yields_data, maturities, dates_dt, model='DNS')")


## Demo: comparativa de los tres modos de τ

In [ ]:
# ── Demo rapida — 3 modos de tau (n_iter pequeno para prototipado) ───────────
resultados_tau = comparar_modos_tau(
    yields_data, maturities, dates_dt,
    model      = 'DNS',
    n_iter     = 1500,
    n_burnin   = 500,
    n_chains   = 2,
    window_size= max(30, T // 5),
    ar1_prop_std = 0.04,
    verbose    = False,
)

# ── Acceder a secuencia tau_t del modo AR(1) ──────────────────────────────────
res_ar1 = resultados_tau['AR(1)']
tau1_t_ar1 = res_ar1['fit']['tau1_t']           # (T,) — media posterior por periodo
tau1_seq   = np.concatenate([ch['tau1_seq']      # (n_samples, T) — todas las muestras
                              for ch in res_ar1['chains']], axis=0)
tau1_ic025 = np.percentile(tau1_seq, 2.5,  axis=0)
tau1_ic975 = np.percentile(tau1_seq, 97.5, axis=0)

print(f"tau1_t AR(1): min={tau1_t_ar1.min():.3f}  "
      f"med={np.median(tau1_t_ar1):.3f}  max={tau1_t_ar1.max():.3f}")
print(f"Rango IC95% medio: [{tau1_ic025.mean():.3f}, {tau1_ic975.mean():.3f}]")

# ── Modo ventana + Opciones A y C ─────────────────────────────────────────────
res_ventana_AC = fit_dns_bayes(
    dates_dt, yields_data, maturities,
    model            = 'DNSS',
    tau_mode         = 'ventana',
    tau_window_size  = max(40, T // 4),
    n_iter           = 2000,
    n_burnin         = 700,
    n_chains         = 2,
    obs_weights      = np.where(maturities >= 10, 2.0, 1.0),
    r_mode           = 'grupos',
    plot             = False,
    verbose          = True,
)
print(f"\nDNSS ventana + AC: RMSE={res_ventana_AC['fit']['rmse_total']*100:.4f}pb")
